# Evaluación final y SHAP con inferencia FP32

Este cuaderno realiza una nueva ejecución final con las configuraciones y los epochs que ya seleccionó el estudio. **No ejecuta Optuna, nested CV ni los diagnósticos de entrenamiento.**

El plan comprende 45 modelos: tres Transformers, tres variantes y cinco semillas. Reutiliza los nueve modelos conservados de semilla 42 y entrena las 36 combinaciones finales restantes. Conserva BF16 durante el entrenamiento; después recarga cada modelo y utiliza la misma función FP32 para el test y SHAP. No selecciona configuraciones, epochs o semillas con el test.

Los pesos de las semillas faltantes corresponden a entrenamientos nuevos, no a una recuperación exacta de los archivos eliminados. Al adoptar este procedimiento se reporta la ejecución completa, aunque algún resultado sea inferior al anterior.

## Antes de ejecutar

- Abrir este cuaderno en la máquina con GPU del entrenamiento y seleccionar el entorno original de los notebooks 04.
- Ejecutar todas las celdas de este cuaderno. No volver a ejecutar los notebooks 04, 05 o 06.
- Se necesitan aproximadamente 26 GB libres para guardar los 45 modelos, las bases y los checkpoints temporales. El cuaderno comprueba el espacio antes de entrenar.
- La salida es `data/reevaluacion_final_fp32_v1/`. Los resultados anteriores se conservan; sus huellas se comprueban al finalizar.
- Si la ejecución se interrumpe, ejecutar de nuevo el mismo cuaderno: reutiliza los modelos finales completados, reanuda checkpoints parciales y conserva las explicaciones SHAP ya calculadas.

El cuaderno contiene una copia congelada de las funciones de entrenamiento originales. Los nuevos entrenamientos descargan el checkpoint base solo si no está en la caché y fijan su revisión para todas las semillas nuevas de cada modelo.


In [ ]:
from pathlib import Path

candidatas = [Path.cwd(), *Path.cwd().parents]
RAIZ_PROYECTO = next((p for p in candidatas if (p / "data" / "resultados_robertuito_grouped_5x3_tpe20").is_dir()), None)
if RAIZ_PROYECTO is None:
    raise FileNotFoundError("Abre este cuaderno desde la carpeta notebooks o la raíz de DeteccionDeSarcasmo.")

# Mantener el mismo nombre para continuar una ejecución interrumpida.
SALIDA_FP32 = RAIZ_PROYECTO / "data" / "reevaluacion_final_fp32_v1"
print("Resultados nuevos en:", SALIDA_FP32)


## 1. Configuraciones y funciones originales congeladas

Esta celda define las fuentes. No ejecuta entrenamiento ni búsquedas.

In [ ]:
FUENTES_MODELOS = {'BERT': {'prefijo': 'beto', 'carpeta': 'resultados_beto_grouped_5x3_tpe20', 'definiciones': 'class DatasetTokenizado(Dataset):\n    def __init__(self, textos, etiquetas):\n        self.encoded = tokenizer(\n            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False\n        )\n        self.labels = np.asarray(etiquetas, dtype=np.int64)\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, i):\n        item = {k: v[i] for k, v in self.encoded.items()}\n        item["labels"] = int(self.labels[i])\n        return item\n\ndef establecer_semilla(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    set_seed(seed)\n\ndef liberar_memoria():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\ndef softmax_np(logits):\n    z = np.asarray(logits, dtype=np.float64)\n    z = z - z.max(axis=1, keepdims=True)\n    e = np.exp(z)\n    return e / e.sum(axis=1, keepdims=True)\n\ndef nll_binaria(probs, labels):\n    """NLL estándar: SIN pesos de clase y SIN label smoothing."""\n    labels = np.asarray(labels, dtype=int)\n    return float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0))))\n\ndef ece_binaria(probs, labels, n_bins=10):\n    """ECE con 10 bins de igual masa sobre la confianza de la clase predicha."""\n    labels = np.asarray(labels, dtype=int)\n    conf = probs.max(axis=1)\n    acierto = (probs.argmax(axis=1) == labels).astype(float)\n    grupos = np.array_split(np.argsort(conf, kind="stable"), n_bins)\n    return float(sum(len(g) / len(conf) * abs(acierto[g].mean() - conf[g].mean())\n                     for g in grupos if len(g)))\n\ndef metricas(prediction):\n    logits, labels = prediction\n    if isinstance(logits, tuple):\n        logits = logits[0]\n    y_pred = np.asarray(logits).argmax(axis=-1)\n    probs = softmax_np(logits)\n    return {\n        "nll": nll_binaria(probs, labels),\n        "ece": ece_binaria(probs, labels),\n        "f1_macro": f1_score(labels, y_pred, average="macro"),\n        "accuracy": accuracy_score(labels, y_pred),\n        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),\n        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),\n    }\n\ndef pesos_balanceados(y):\n    classes = np.array([0, 1])\n    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))\n    return torch.tensor(w, dtype=torch.float32)\n\ndef congelar_fraccion_inferior(model, fraccion):\n    """congela embeddings + la fracción inferior de capas del encoder."""\n    if not fraccion:\n        return []\n    n_total = 1 + max(int(m.group(1)) for n, _ in model.named_parameters()\n                      if (m := PATRON_CAPA.search(n)))\n    n_congeladas = max(1, round(n_total * float(fraccion)))\n    capas = set()\n    for nombre, p in model.named_parameters():\n        m = PATRON_CAPA.search(nombre)\n        if "embeddings" in nombre or (m and int(m.group(1)) < n_congeladas):\n            p.requires_grad = False\n            if m:\n                capas.add(int(m.group(1)))\n    return sorted(capas)\n\ndef crear_modelo(seed, classifier_dropout=0.2, cfg=None):\n    """`cfg` aporta `dropout_interno` y `congelar_frac`."""\n    cfg = cfg or {}\n    establecer_semilla(seed)\n    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, redirect_stdout(\n        log\n    ), redirect_stderr(log):\n        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)\n        if config.model_type != "bert":\n            raise ValueError(f"Se esperaba arquitectura BERT, recibida: {config.model_type}")\n        # BERT: dropout de la cabeza de clasificación.\n        config.classifier_dropout = float(classifier_dropout)\n        # dropout interno (oculto y de atención) del encoder y los embeddings.\n        dropout_interno = float(cfg.get("dropout_interno", getattr(config, "hidden_dropout_prob")))\n        config.hidden_dropout_prob = dropout_interno\n        config.attention_probs_dropout_prob = dropout_interno\n        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)\n    congelar_fraccion_inferior(model, cfg.get("congelar_frac", 0.0))\n    return model\n\nclass ClasificadorTrainer(Trainer):\n    """pérdida ponderada opcional + label smoothing (solo en train) + LR decay por capas\n    + enmascarado aleatorio de tokens."""\n\n    def __init__(self, *args, class_weights=None, label_smoothing=0.0, lr_decay=1.0,\n                 mascara_tokens=0.0, **kwargs):\n        self.class_weights = class_weights\n        self.label_smoothing = float(label_smoothing)\n        self.lr_decay = float(lr_decay)\n        self.mascara_tokens = float(mascara_tokens)\n        self._especiales = torch.tensor(sorted(set(tokenizer.all_special_ids)))\n        super().__init__(*args, **kwargs)\n\n    def _enmascarar(self, inputs):\n        """Reemplaza al azar `mascara_tokens` de los tokens (no especiales, no padding) por el\n        token de máscara. Validación, refit-evaluación y test leen siempre el texto completo."""\n        ids = inputs["input_ids"].clone()\n        candidatos = ~torch.isin(ids, self._especiales.to(ids.device))\n        if "attention_mask" in inputs:\n            candidatos &= inputs["attention_mask"].bool()\n        sorteo = torch.rand(ids.shape, device=ids.device) < self.mascara_tokens\n        ids[candidatos & sorteo] = tokenizer.mask_token_id\n        return {**inputs, "input_ids": ids}\n\n    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):\n        if model.training and self.mascara_tokens > 0:\n            inputs = self._enmascarar(inputs)\n        labels = inputs["labels"]\n        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})\n        weights = (\n            self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None\n        )\n        # Smoothing SOLO al entrenar. En evaluación la pérdida (eval_loss) sigue ponderada por clase;\n        # la NLL sin pesos de clase se calcula aparte en metricas().\n        ls = self.label_smoothing if model.training else 0.0\n        loss = F.cross_entropy(outputs.logits, labels, weight=weights, label_smoothing=ls)\n        return (loss, outputs) if return_outputs else loss\n\n    def create_optimizer(self, *args, **kwargs):\n        if self.optimizer is not None:\n            return self.optimizer\n        if self.lr_decay >= 1.0:\n            return super().create_optimizer(*args, **kwargs)\n        # Capas: BERT/RoBERTa `encoder.layer.N`, DistilBERT `transformer.layer.N`.\n        patron = re.compile(r"\\.layer\\.(\\d+)\\.")\n        n_capas = 1 + max(\n            int(m.group(1)) for n, _ in self.model.named_parameters() if (m := patron.search(n))\n        )\n        base_lr, wd = self.args.learning_rate, self.args.weight_decay\n        grupos = {}\n        for nombre, p in self.model.named_parameters():\n            if not p.requires_grad:\n                continue\n            m = patron.search(nombre)\n            if m:\n                profundidad = n_capas - int(m.group(1))  # última capa = 1\n            elif "embeddings" in nombre:\n                profundidad = n_capas + 1  # embeddings = LR más bajo\n            else:\n                profundidad = 0  # cabeza de clasificación / pooler = LR base\n            sin_wd = nombre.endswith("bias") or "LayerNorm" in nombre or "layer_norm" in nombre\n            clave = (base_lr * self.lr_decay**profundidad, 0.0 if sin_wd else wd)\n            grupos.setdefault(clave, []).append(p)\n        self.optimizer = torch.optim.AdamW(\n            [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in grupos.items()]\n        )\n        return self.optimizer\n\nclass ParadaTemprana(TrainerCallback):\n    """Early stopping por F1-Macro de validación sin guardar checkpoints (Inner Folds)."""\n\n    def __init__(self, paciencia=2):\n        self.paciencia, self.mejor, self.sin_mejora = int(paciencia), -np.inf, 0\n\n    def on_evaluate(self, args, state, control, metrics=None, **kwargs):\n        f1 = (metrics or {}).get("eval_f1_macro")\n        if f1 is None:\n            return control\n        if f1 > self.mejor:\n            self.mejor, self.sin_mejora = f1, 0\n        else:\n            self.sin_mejora += 1\n            if self.sin_mejora >= self.paciencia:\n                control.should_training_stop = True\n        return control\n\nclass DetenerEnEpoca(TrainerCallback):\n    """Refit/test con el MISMO scheduler que la búsqueda: se detiene en la época elegida."""\n\n    def __init__(self, epocas):\n        self.epocas = int(epocas)\n\n    def on_epoch_end(self, args, state, control, **kwargs):\n        if state.epoch is not None and state.epoch >= self.epocas - 1e-6:\n            control.should_training_stop = True\n        return control\n\ndef argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):\n    params = {\n        "output_dir": str(carpeta),\n        "learning_rate": cfg["learning_rate"],\n        "per_device_train_batch_size": cfg["batch_size"],\n        "per_device_eval_batch_size": cfg["batch_size"],\n        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],\n        "weight_decay": cfg["weight_decay"],\n        "optim": "adamw_torch",\n        # Trainer recoge loss por época en state.log_history;\n        # PrinterCallback se elimina para no saturar VS Code.\n        "logging_strategy": "epoch",\n        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.\n        "load_best_model_at_end": bool(con_validacion),\n        "save_total_limit": 2,  # último + mejor (si son diferentes)\n        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.\n        "report_to": "none",\n        "disable_tqdm": True,\n        # Necesario para que EarlyStoppingCallback conserve su contador\n        # al reanudar desde un checkpoint.\n        "restore_callback_states_from_checkpoint": True,\n        "dataloader_num_workers": 0,\n        "seed": seed,\n        "data_seed": seed,\n        "bf16": USE_BF16,\n        "fp16": USE_FP16,\n    }\n    if con_validacion:\n        # Inner Folds sin checkpoints: el modelo interno se descarta y F1/mejor época\n        # se leen de la curva. El early stopping lo hace ParadaTemprana.\n        params.update({"save_strategy": "no", "load_best_model_at_end": False})\n    sig = inspect.signature(TrainingArguments.__init__).parameters\n    if "eval_strategy" in sig:\n        params["eval_strategy"] = "epoch" if con_validacion else "no"\n    elif "evaluation_strategy" in sig:\n        params["evaluation_strategy"] = "epoch" if con_validacion else "no"\n    else:\n        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")\n    # warmup explícito (transformers 5: `warmup_steps` acepta una fracción en [0, 1)).\n    if "warmup_ratio" in sig:\n        params["warmup_ratio"] = float(WARMUP_FRACCION)\n    else:\n        params["warmup_steps"] = float(WARMUP_FRACCION)\n    # nunca descartar argumentos en silencio.\n    descartados = sorted(k for k in params if k not in sig)\n    if descartados:\n        raise RuntimeError(f"TrainingArguments no acepta {descartados} en esta versión de transformers.")\n    return TrainingArguments(**params)', 'notebook_fuente': '04a_BERT.ipynb', 'sha256_notebook_fuente': 'fb4e2faa8cd76c836a7714f47a0bb12e7299f55c861687663852866a29aae6ec', 'codigo_flujo_sha256': '7565cd917834e14b3fb92a310a8e41de6d917e103f86af941bc02758a95fea26', 'metadatos': {'mx': {'modelo_base': 'dccuchile/bert-base-spanish-wwm-cased', 'variante': 'mx', 'seed': 42, 'config_id': 'P_l2_b16_e6_w1_b_d1_s1_r9_i2_c0_m15', 'configuracion': {'preprocessing': 'PYSENTIMIENTO', 'learning_rate': 2e-05, 'batch_size': 16, 'num_train_epochs': 6, 'weight_decay': 0.01, 'loss_mode': 'balanced', 'classifier_dropout': 0.1, 'label_smoothing': 0.1, 'lr_decay': 0.9, 'dropout_interno': 0.2, 'congelar_frac': 0.0, 'mascara_tokens': 0.15, 'refit_epochs': 5}, 'epocas_entrenadas': 5.0, 'n_train': 2399, 'csv_train': '..\\data\\transformers\\train_transformers_mx.csv', 'sha256_csv_train': '8728c2bfe70281f2bdc27b0993f7157e094ed6494d015518a1acbfb20f5f9d7d', 'columna_texto': 'MESSAGE_TRANSFORMER', 'max_length': 128, 'temperatura': 1.496356191437287, 'etiquetas': {'0': 'NO_IRONICO', '1': 'IRONICO'}, 'versiones': {'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'optuna': '5.0.0', 'numpy': '1.26.4', 'pandas': '3.0.3'}, 'fecha': '2026-10-28T18:00:47'}, 'es': {'modelo_base': 'dccuchile/bert-base-spanish-wwm-cased', 'variante': 'es', 'seed': 42, 'config_id': 'P_l2_b16_e6_w10_b_d1_s1_r9_i2_c0_m15', 'configuracion': {'preprocessing': 'PYSENTIMIENTO', 'learning_rate': 2e-05, 'batch_size': 16, 'num_train_epochs': 6, 'weight_decay': 0.1, 'loss_mode': 'balanced', 'classifier_dropout': 0.1, 'label_smoothing': 0.1, 'lr_decay': 0.9, 'dropout_interno': 0.2, 'congelar_frac': 0.0, 'mascara_tokens': 0.15, 'refit_epochs': 6}, 'epocas_entrenadas': 6.0, 'n_train': 2397, 'csv_train': '..\\data\\transformers\\train_transformers_es.csv', 'sha256_csv_train': '4526a9bedeceb6cf9b0f95cb7f2023bb21fc661245c9d69d822be682902e6524', 'columna_texto': 'MESSAGE_TRANSFORMER', 'max_length': 128, 'temperatura': 1.1332381911864717, 'etiquetas': {'0': 'NO_IRONICO', '1': 'IRONICO'}, 'versiones': {'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'optuna': '5.0.0', 'numpy': '1.26.4', 'pandas': '3.0.3'}, 'fecha': '2026-10-28T18:01:37'}, 'cu': {'modelo_base': 'dccuchile/bert-base-spanish-wwm-cased', 'variante': 'cu', 'seed': 42, 'config_id': 'P_l2_b16_e6_w10_b_d3_s1_r9_i2_c0_m15', 'configuracion': {'preprocessing': 'PYSENTIMIENTO', 'learning_rate': 2e-05, 'batch_size': 16, 'num_train_epochs': 6, 'weight_decay': 0.1, 'loss_mode': 'balanced', 'classifier_dropout': 0.3, 'label_smoothing': 0.1, 'lr_decay': 0.9, 'dropout_interno': 0.2, 'congelar_frac': 0.0, 'mascara_tokens': 0.15, 'refit_epochs': 6}, 'epocas_entrenadas': 6.0, 'n_train': 2400, 'csv_train': '..\\data\\transformers\\train_transformers_cu.csv', 'sha256_csv_train': 'c0f434b52cae1ef5b8aefbb58da7925248822f889d248b6b0ac1adf809fba00f', 'columna_texto': 'MESSAGE_TRANSFORMER', 'max_length': 128, 'temperatura': 1.1530969240758577, 'etiquetas': {'0': 'NO_IRONICO', '1': 'IRONICO'}, 'versiones': {'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'optuna': '5.0.0', 'numpy': '1.26.4', 'pandas': '3.0.3'}, 'fecha': '2026-10-28T18:02:27'}}}, 'DistilBERT': {'prefijo': 'distilbeto', 'carpeta': 'resultados_distilbeto_grouped_5x3_tpe20', 'definiciones': 'class DatasetTokenizado(Dataset):\n    def __init__(self, textos, etiquetas):\n        self.encoded = tokenizer(\n            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False\n        )\n        self.labels = np.asarray(etiquetas, dtype=np.int64)\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, i):\n        item = {k: v[i] for k, v in self.encoded.items()}\n        item["labels"] = int(self.labels[i])\n        return item\n\ndef establecer_semilla(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    set_seed(seed)\n\ndef liberar_memoria():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\ndef softmax_np(logits):\n    z = np.asarray(logits, dtype=np.float64)\n    z = z - z.max(axis=1, keepdims=True)\n    e = np.exp(z)\n    return e / e.sum(axis=1, keepdims=True)\n\ndef nll_binaria(probs, labels):\n    """NLL estándar: SIN pesos de clase y SIN label smoothing."""\n    labels = np.asarray(labels, dtype=int)\n    return float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0))))\n\ndef ece_binaria(probs, labels, n_bins=10):\n    """ECE con 10 bins de igual masa sobre la confianza de la clase predicha."""\n    labels = np.asarray(labels, dtype=int)\n    conf = probs.max(axis=1)\n    acierto = (probs.argmax(axis=1) == labels).astype(float)\n    grupos = np.array_split(np.argsort(conf, kind="stable"), n_bins)\n    return float(sum(len(g) / len(conf) * abs(acierto[g].mean() - conf[g].mean())\n                     for g in grupos if len(g)))\n\ndef metricas(prediction):\n    logits, labels = prediction\n    if isinstance(logits, tuple):\n        logits = logits[0]\n    y_pred = np.asarray(logits).argmax(axis=-1)\n    probs = softmax_np(logits)\n    return {\n        "nll": nll_binaria(probs, labels),\n        "ece": ece_binaria(probs, labels),\n        "f1_macro": f1_score(labels, y_pred, average="macro"),\n        "accuracy": accuracy_score(labels, y_pred),\n        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),\n        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),\n    }\n\ndef pesos_balanceados(y):\n    classes = np.array([0, 1])\n    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))\n    return torch.tensor(w, dtype=torch.float32)\n\ndef congelar_fraccion_inferior(model, fraccion):\n    """congela embeddings + la fracción inferior de capas del encoder."""\n    if not fraccion:\n        return []\n    n_total = 1 + max(int(m.group(1)) for n, _ in model.named_parameters()\n                      if (m := PATRON_CAPA.search(n)))\n    n_congeladas = max(1, round(n_total * float(fraccion)))\n    capas = set()\n    for nombre, p in model.named_parameters():\n        m = PATRON_CAPA.search(nombre)\n        if "embeddings" in nombre or (m and int(m.group(1)) < n_congeladas):\n            p.requires_grad = False\n            if m:\n                capas.add(int(m.group(1)))\n    return sorted(capas)\n\ndef crear_modelo(seed, classifier_dropout=0.2, cfg=None):\n    """`cfg` aporta `dropout_interno` y `congelar_frac`."""\n    cfg = cfg or {}\n    establecer_semilla(seed)\n    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, redirect_stdout(\n        log\n    ), redirect_stderr(log):\n        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)\n        if config.model_type != "distilbert":\n            raise ValueError(f"Se esperaba arquitectura DistilBERT, recibida: {config.model_type}")\n        # DistilBERT guarda el dropout de la cabeza de clasificación en `seq_classif_dropout`.\n        config.seq_classif_dropout = float(classifier_dropout)\n        # dropout interno (oculto y de atención) del encoder y los embeddings.\n        dropout_interno = float(cfg.get("dropout_interno", getattr(config, "dropout")))\n        config.dropout = dropout_interno\n        config.attention_dropout = dropout_interno\n        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)\n    congelar_fraccion_inferior(model, cfg.get("congelar_frac", 0.0))\n    return model\n\nclass ClasificadorTrainer(Trainer):\n    """pérdida ponderada opcional + label smoothing (solo en train) + LR decay por capas\n    + enmascarado aleatorio de tokens."""\n\n    def __init__(self, *args, class_weights=None, label_smoothing=0.0, lr_decay=1.0,\n                 mascara_tokens=0.0, **kwargs):\n        self.class_weights = class_weights\n        self.label_smoothing = float(label_smoothing)\n        self.lr_decay = float(lr_decay)\n        self.mascara_tokens = float(mascara_tokens)\n        self._especiales = torch.tensor(sorted(set(tokenizer.all_special_ids)))\n        super().__init__(*args, **kwargs)\n\n    def _enmascarar(self, inputs):\n        """Reemplaza al azar `mascara_tokens` de los tokens (no especiales, no padding) por el\n        token de máscara. Validación, refit-evaluación y test leen siempre el texto completo."""\n        ids = inputs["input_ids"].clone()\n        candidatos = ~torch.isin(ids, self._especiales.to(ids.device))\n        if "attention_mask" in inputs:\n            candidatos &= inputs["attention_mask"].bool()\n        sorteo = torch.rand(ids.shape, device=ids.device) < self.mascara_tokens\n        ids[candidatos & sorteo] = tokenizer.mask_token_id\n        return {**inputs, "input_ids": ids}\n\n    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):\n        if model.training and self.mascara_tokens > 0:\n            inputs = self._enmascarar(inputs)\n        labels = inputs["labels"]\n        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})\n        weights = (\n            self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None\n        )\n        # Smoothing SOLO al entrenar. En evaluación la pérdida (eval_loss) sigue ponderada por clase;\n        # la NLL sin pesos de clase se calcula aparte en metricas().\n        ls = self.label_smoothing if model.training else 0.0\n        loss = F.cross_entropy(outputs.logits, labels, weight=weights, label_smoothing=ls)\n        return (loss, outputs) if return_outputs else loss\n\n    def create_optimizer(self, *args, **kwargs):\n        if self.optimizer is not None:\n            return self.optimizer\n        if self.lr_decay >= 1.0:\n            return super().create_optimizer(*args, **kwargs)\n        # Capas: BERT/RoBERTa `encoder.layer.N`, DistilBERT `transformer.layer.N`.\n        patron = re.compile(r"\\.layer\\.(\\d+)\\.")\n        n_capas = 1 + max(\n            int(m.group(1)) for n, _ in self.model.named_parameters() if (m := patron.search(n))\n        )\n        base_lr, wd = self.args.learning_rate, self.args.weight_decay\n        grupos = {}\n        for nombre, p in self.model.named_parameters():\n            if not p.requires_grad:\n                continue\n            m = patron.search(nombre)\n            if m:\n                profundidad = n_capas - int(m.group(1))  # última capa = 1\n            elif "embeddings" in nombre:\n                profundidad = n_capas + 1  # embeddings = LR más bajo\n            else:\n                profundidad = 0  # cabeza de clasificación / pooler = LR base\n            sin_wd = nombre.endswith("bias") or "LayerNorm" in nombre or "layer_norm" in nombre\n            clave = (base_lr * self.lr_decay**profundidad, 0.0 if sin_wd else wd)\n            grupos.setdefault(clave, []).append(p)\n        self.optimizer = torch.optim.AdamW(\n            [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in grupos.items()]\n        )\n        return self.optimizer\n\nclass ParadaTemprana(TrainerCallback):\n    """Early stopping por F1-Macro de validación sin guardar checkpoints (Inner Folds)."""\n\n    def __init__(self, paciencia=2):\n        self.paciencia, self.mejor, self.sin_mejora = int(paciencia), -np.inf, 0\n\n    def on_evaluate(self, args, state, control, metrics=None, **kwargs):\n        f1 = (metrics or {}).get("eval_f1_macro")\n        if f1 is None:\n            return control\n        if f1 > self.mejor:\n            self.mejor, self.sin_mejora = f1, 0\n        else:\n            self.sin_mejora += 1\n            if self.sin_mejora >= self.paciencia:\n                control.should_training_stop = True\n        return control\n\nclass DetenerEnEpoca(TrainerCallback):\n    """Refit/test con el MISMO scheduler que la búsqueda: se detiene en la época elegida."""\n\n    def __init__(self, epocas):\n        self.epocas = int(epocas)\n\n    def on_epoch_end(self, args, state, control, **kwargs):\n        if state.epoch is not None and state.epoch >= self.epocas - 1e-6:\n            control.should_training_stop = True\n        return control\n\ndef argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):\n    params = {\n        "output_dir": str(carpeta),\n        "learning_rate": cfg["learning_rate"],\n        "per_device_train_batch_size": cfg["batch_size"],\n        "per_device_eval_batch_size": cfg["batch_size"],\n        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],\n        "weight_decay": cfg["weight_decay"],\n        "optim": "adamw_torch",\n        # Trainer recoge loss por época en state.log_history;\n        # PrinterCallback se elimina para no saturar VS Code.\n        "logging_strategy": "epoch",\n        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.\n        "load_best_model_at_end": bool(con_validacion),\n        "save_total_limit": 2,  # último + mejor (si son diferentes)\n        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.\n        "report_to": "none",\n        "disable_tqdm": True,\n        # Necesario para que EarlyStoppingCallback conserve su contador\n        # al reanudar desde un checkpoint.\n        "restore_callback_states_from_checkpoint": True,\n        "dataloader_num_workers": 0,\n        "seed": seed,\n        "data_seed": seed,\n        "bf16": USE_BF16,\n        "fp16": USE_FP16,\n    }\n    if con_validacion:\n        # Inner Folds sin checkpoints: el modelo interno se descarta y F1/mejor época\n        # se leen de la curva. El early stopping lo hace ParadaTemprana.\n        params.update({"save_strategy": "no", "load_best_model_at_end": False})\n    sig = inspect.signature(TrainingArguments.__init__).parameters\n    if "eval_strategy" in sig:\n        params["eval_strategy"] = "epoch" if con_validacion else "no"\n    elif "evaluation_strategy" in sig:\n        params["evaluation_strategy"] = "epoch" if con_validacion else "no"\n    else:\n        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")\n    # warmup explícito (transformers 5: `warmup_steps` acepta una fracción en [0, 1)).\n    if "warmup_ratio" in sig:\n        params["warmup_ratio"] = float(WARMUP_FRACCION)\n    else:\n        params["warmup_steps"] = float(WARMUP_FRACCION)\n    # nunca descartar argumentos en silencio.\n    descartados = sorted(k for k in params if k not in sig)\n    if descartados:\n        raise RuntimeError(f"TrainingArguments no acepta {descartados} en esta versión de transformers.")\n    return TrainingArguments(**params)', 'notebook_fuente': '04b_DistilBERT.ipynb', 'sha256_notebook_fuente': '87b3acb074cf772115bcaf60348a2fb2152f5b19ca11a2844dfa2c58b7ca3d6f', 'codigo_flujo_sha256': '7565cd917834e14b3fb92a310a8e41de6d917e103f86af941bc02758a95fea26', 'metadatos': {'mx': {'modelo_base': 'dccuchile/distilbert-base-spanish-uncased', 'variante': 'mx', 'seed': 42, 'config_id': 'P_l2_b16_e6_w1_b_d3_s1_r9_i2_c0_m15', 'configuracion': {'preprocessing': 'PYSENTIMIENTO', 'learning_rate': 2e-05, 'batch_size': 16, 'num_train_epochs': 6, 'weight_decay': 0.01, 'loss_mode': 'balanced', 'classifier_dropout': 0.3, 'label_smoothing': 0.1, 'lr_decay': 0.9, 'dropout_interno': 0.2, 'congelar_frac': 0.0, 'mascara_tokens': 0.15, 'refit_epochs': 5}, 'epocas_entrenadas': 5.0, 'n_train': 2399, 'csv_train': '..\\data\\transformers\\train_transformers_mx.csv', 'sha256_csv_train': '8728c2bfe70281f2bdc27b0993f7157e094ed6494d015518a1acbfb20f5f9d7d', 'columna_texto': 'MESSAGE_TRANSFORMER', 'max_length': 128, 'temperatura': 1.104089513673812, 'etiquetas': {'0': 'NO_IRONICO', '1': 'IRONICO'}, 'versiones': {'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'optuna': '5.0.0', 'numpy': '1.26.4', 'pandas': '3.0.3'}, 'fecha': '2026-10-28T19:43:32'}, 'es': {'modelo_base': 'dccuchile/distilbert-base-spanish-uncased', 'variante': 'es', 'seed': 42, 'config_id': 'P_l2_b16_e6_w1_b_d2_s1_r9_i2_c0_m15', 'configuracion': {'preprocessing': 'PYSENTIMIENTO', 'learning_rate': 2e-05, 'batch_size': 16, 'num_train_epochs': 6, 'weight_decay': 0.01, 'loss_mode': 'balanced', 'classifier_dropout': 0.2, 'label_smoothing': 0.1, 'lr_decay': 0.9, 'dropout_interno': 0.2, 'congelar_frac': 0.0, 'mascara_tokens': 0.15, 'refit_epochs': 5}, 'epocas_entrenadas': 5.0, 'n_train': 2397, 'csv_train': '..\\data\\transformers\\train_transformers_es.csv', 'sha256_csv_train': '4526a9bedeceb6cf9b0f95cb7f2023bb21fc661245c9d69d822be682902e6524', 'columna_texto': 'MESSAGE_TRANSFORMER', 'max_length': 128, 'temperatura': 0.9776693610686636, 'etiquetas': {'0': 'NO_IRONICO', '1': 'IRONICO'}, 'versiones': {'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'optuna': '5.0.0', 'numpy': '1.26.4', 'pandas': '3.0.3'}, 'fecha': '2026-10-28T19:45:47'}, 'cu': {'modelo_base': 'dccuchile/distilbert-base-spanish-uncased', 'variante': 'cu', 'seed': 42, 'config_id': 'P_l2_b16_e6_w10_b_d1_s1_r9_i2_c0_m15', 'configuracion': {'preprocessing': 'PYSENTIMIENTO', 'learning_rate': 2e-05, 'batch_size': 16, 'num_train_epochs': 6, 'weight_decay': 0.1, 'loss_mode': 'balanced', 'classifier_dropout': 0.1, 'label_smoothing': 0.1, 'lr_decay': 0.9, 'dropout_interno': 0.2, 'congelar_frac': 0.0, 'mascara_tokens': 0.15, 'refit_epochs': 6}, 'epocas_entrenadas': 6.0, 'n_train': 2400, 'csv_train': '..\\data\\transformers\\train_transformers_cu.csv', 'sha256_csv_train': 'ab8a76cf77a4189ce5921d3edb6dc174d6b9c2e090ce7654f376ee92e3b1b05a', 'columna_texto': 'MESSAGE_TRANSFORMER', 'max_length': 128, 'temperatura': 1.0034804647580498, 'etiquetas': {'0': 'NO_IRONICO', '1': 'IRONICO'}, 'versiones': {'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'optuna': '5.0.0', 'numpy': '1.26.4', 'pandas': '3.0.3'}, 'fecha': '2026-10-28T19:48:06'}}}, 'RoBERTuito': {'prefijo': 'robertuito', 'carpeta': 'resultados_robertuito_grouped_5x3_tpe20', 'definiciones': 'class DatasetTokenizado(Dataset):\n    def __init__(self, textos, etiquetas):\n        self.encoded = tokenizer(\n            [str(t) for t in textos], truncation=True, max_length=MAX_LENGTH, padding=False\n        )\n        self.labels = np.asarray(etiquetas, dtype=np.int64)\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, i):\n        item = {k: v[i] for k, v in self.encoded.items()}\n        item["labels"] = int(self.labels[i])\n        return item\n\ndef establecer_semilla(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    set_seed(seed)\n\ndef liberar_memoria():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\ndef softmax_np(logits):\n    z = np.asarray(logits, dtype=np.float64)\n    z = z - z.max(axis=1, keepdims=True)\n    e = np.exp(z)\n    return e / e.sum(axis=1, keepdims=True)\n\ndef nll_binaria(probs, labels):\n    """NLL estándar: SIN pesos de clase y SIN label smoothing."""\n    labels = np.asarray(labels, dtype=int)\n    return float(-np.mean(np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0))))\n\ndef ece_binaria(probs, labels, n_bins=10):\n    """ECE con 10 bins de igual masa sobre la confianza de la clase predicha."""\n    labels = np.asarray(labels, dtype=int)\n    conf = probs.max(axis=1)\n    acierto = (probs.argmax(axis=1) == labels).astype(float)\n    grupos = np.array_split(np.argsort(conf, kind="stable"), n_bins)\n    return float(sum(len(g) / len(conf) * abs(acierto[g].mean() - conf[g].mean())\n                     for g in grupos if len(g)))\n\ndef metricas(prediction):\n    logits, labels = prediction\n    if isinstance(logits, tuple):\n        logits = logits[0]\n    y_pred = np.asarray(logits).argmax(axis=-1)\n    probs = softmax_np(logits)\n    return {\n        "nll": nll_binaria(probs, labels),\n        "ece": ece_binaria(probs, labels),\n        "f1_macro": f1_score(labels, y_pred, average="macro"),\n        "accuracy": accuracy_score(labels, y_pred),\n        "precision_macro": precision_score(labels, y_pred, average="macro", zero_division=0),\n        "recall_macro": recall_score(labels, y_pred, average="macro", zero_division=0),\n    }\n\ndef pesos_balanceados(y):\n    classes = np.array([0, 1])\n    w = compute_class_weight("balanced", classes=classes, y=np.asarray(y))\n    return torch.tensor(w, dtype=torch.float32)\n\ndef congelar_fraccion_inferior(model, fraccion):\n    """congela embeddings + la fracción inferior de capas del encoder."""\n    if not fraccion:\n        return []\n    n_total = 1 + max(int(m.group(1)) for n, _ in model.named_parameters()\n                      if (m := PATRON_CAPA.search(n)))\n    n_congeladas = max(1, round(n_total * float(fraccion)))\n    capas = set()\n    for nombre, p in model.named_parameters():\n        m = PATRON_CAPA.search(nombre)\n        if "embeddings" in nombre or (m and int(m.group(1)) < n_congeladas):\n            p.requires_grad = False\n            if m:\n                capas.add(int(m.group(1)))\n    return sorted(capas)\n\ndef crear_modelo(seed, classifier_dropout=0.2, cfg=None):\n    """`cfg` aporta `dropout_interno` y `congelar_frac`."""\n    cfg = cfg or {}\n    establecer_semilla(seed)\n    with (RUN_DIR / "carga_modelos.log").open("a", encoding="utf-8") as log, redirect_stdout(\n        log\n    ), redirect_stderr(log):\n        config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)\n        if config.model_type != "roberta":\n            raise ValueError(f"Se esperaba arquitectura RoBERTa, recibida: {config.model_type}")\n        # RoBERTa: dropout de la cabeza de clasificación.\n        config.classifier_dropout = float(classifier_dropout)\n        # dropout interno (oculto y de atención) del encoder y los embeddings.\n        dropout_interno = float(cfg.get("dropout_interno", getattr(config, "hidden_dropout_prob")))\n        config.hidden_dropout_prob = dropout_interno\n        config.attention_probs_dropout_prob = dropout_interno\n        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)\n    congelar_fraccion_inferior(model, cfg.get("congelar_frac", 0.0))\n    return model\n\nclass ClasificadorTrainer(Trainer):\n    """pérdida ponderada opcional + label smoothing (solo en train) + LR decay por capas\n    + enmascarado aleatorio de tokens."""\n\n    def __init__(self, *args, class_weights=None, label_smoothing=0.0, lr_decay=1.0,\n                 mascara_tokens=0.0, **kwargs):\n        self.class_weights = class_weights\n        self.label_smoothing = float(label_smoothing)\n        self.lr_decay = float(lr_decay)\n        self.mascara_tokens = float(mascara_tokens)\n        self._especiales = torch.tensor(sorted(set(tokenizer.all_special_ids)))\n        super().__init__(*args, **kwargs)\n\n    def _enmascarar(self, inputs):\n        """Reemplaza al azar `mascara_tokens` de los tokens (no especiales, no padding) por el\n        token de máscara. Validación, refit-evaluación y test leen siempre el texto completo."""\n        ids = inputs["input_ids"].clone()\n        candidatos = ~torch.isin(ids, self._especiales.to(ids.device))\n        if "attention_mask" in inputs:\n            candidatos &= inputs["attention_mask"].bool()\n        sorteo = torch.rand(ids.shape, device=ids.device) < self.mascara_tokens\n        ids[candidatos & sorteo] = tokenizer.mask_token_id\n        return {**inputs, "input_ids": ids}\n\n    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):\n        if model.training and self.mascara_tokens > 0:\n            inputs = self._enmascarar(inputs)\n        labels = inputs["labels"]\n        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})\n        weights = (\n            self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None\n        )\n        # Smoothing SOLO al entrenar. En evaluación la pérdida (eval_loss) sigue ponderada por clase;\n        # la NLL sin pesos de clase se calcula aparte en metricas().\n        ls = self.label_smoothing if model.training else 0.0\n        loss = F.cross_entropy(outputs.logits, labels, weight=weights, label_smoothing=ls)\n        return (loss, outputs) if return_outputs else loss\n\n    def create_optimizer(self, *args, **kwargs):\n        if self.optimizer is not None:\n            return self.optimizer\n        if self.lr_decay >= 1.0:\n            return super().create_optimizer(*args, **kwargs)\n        # Capas: BERT/RoBERTa `encoder.layer.N`, DistilBERT `transformer.layer.N`.\n        patron = re.compile(r"\\.layer\\.(\\d+)\\.")\n        n_capas = 1 + max(\n            int(m.group(1)) for n, _ in self.model.named_parameters() if (m := patron.search(n))\n        )\n        base_lr, wd = self.args.learning_rate, self.args.weight_decay\n        grupos = {}\n        for nombre, p in self.model.named_parameters():\n            if not p.requires_grad:\n                continue\n            m = patron.search(nombre)\n            if m:\n                profundidad = n_capas - int(m.group(1))  # última capa = 1\n            elif "embeddings" in nombre:\n                profundidad = n_capas + 1  # embeddings = LR más bajo\n            else:\n                profundidad = 0  # cabeza de clasificación / pooler = LR base\n            sin_wd = nombre.endswith("bias") or "LayerNorm" in nombre or "layer_norm" in nombre\n            clave = (base_lr * self.lr_decay**profundidad, 0.0 if sin_wd else wd)\n            grupos.setdefault(clave, []).append(p)\n        self.optimizer = torch.optim.AdamW(\n            [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in grupos.items()]\n        )\n        return self.optimizer\n\nclass ParadaTemprana(TrainerCallback):\n    """Early stopping por F1-Macro de validación sin guardar checkpoints (Inner Folds)."""\n\n    def __init__(self, paciencia=2):\n        self.paciencia, self.mejor, self.sin_mejora = int(paciencia), -np.inf, 0\n\n    def on_evaluate(self, args, state, control, metrics=None, **kwargs):\n        f1 = (metrics or {}).get("eval_f1_macro")\n        if f1 is None:\n            return control\n        if f1 > self.mejor:\n            self.mejor, self.sin_mejora = f1, 0\n        else:\n            self.sin_mejora += 1\n            if self.sin_mejora >= self.paciencia:\n                control.should_training_stop = True\n        return control\n\nclass DetenerEnEpoca(TrainerCallback):\n    """Refit/test con el MISMO scheduler que la búsqueda: se detiene en la época elegida."""\n\n    def __init__(self, epocas):\n        self.epocas = int(epocas)\n\n    def on_epoch_end(self, args, state, control, **kwargs):\n        if state.epoch is not None and state.epoch >= self.epocas - 1e-6:\n            control.should_training_stop = True\n        return control\n\ndef argumentos_entrenamiento(carpeta, cfg, seed, con_validacion=True, epochs=None):\n    params = {\n        "output_dir": str(carpeta),\n        "learning_rate": cfg["learning_rate"],\n        "per_device_train_batch_size": cfg["batch_size"],\n        "per_device_eval_batch_size": cfg["batch_size"],\n        "num_train_epochs": epochs if epochs is not None else cfg["num_train_epochs"],\n        "weight_decay": cfg["weight_decay"],\n        "optim": "adamw_torch",\n        # Trainer recoge loss por época en state.log_history;\n        # PrinterCallback se elimina para no saturar VS Code.\n        "logging_strategy": "epoch",\n        "save_strategy": "epoch",  # Inner y refit: reinicio desde última época completa.\n        "load_best_model_at_end": bool(con_validacion),\n        "save_total_limit": 2,  # último + mejor (si son diferentes)\n        "save_only_model": False,  # Estado optimizer/scheduler requerido para reanudar.\n        "report_to": "none",\n        "disable_tqdm": True,\n        # Necesario para que EarlyStoppingCallback conserve su contador\n        # al reanudar desde un checkpoint.\n        "restore_callback_states_from_checkpoint": True,\n        "dataloader_num_workers": 0,\n        "seed": seed,\n        "data_seed": seed,\n        "bf16": USE_BF16,\n        "fp16": USE_FP16,\n    }\n    if con_validacion:\n        # Inner Folds sin checkpoints: el modelo interno se descarta y F1/mejor época\n        # se leen de la curva. El early stopping lo hace ParadaTemprana.\n        params.update({"save_strategy": "no", "load_best_model_at_end": False})\n    sig = inspect.signature(TrainingArguments.__init__).parameters\n    if "eval_strategy" in sig:\n        params["eval_strategy"] = "epoch" if con_validacion else "no"\n    elif "evaluation_strategy" in sig:\n        params["evaluation_strategy"] = "epoch" if con_validacion else "no"\n    else:\n        raise RuntimeError("Esta versión de Transformers no admite evaluación por época.")\n    # warmup explícito (transformers 5: `warmup_steps` acepta una fracción en [0, 1)).\n    if "warmup_ratio" in sig:\n        params["warmup_ratio"] = float(WARMUP_FRACCION)\n    else:\n        params["warmup_steps"] = float(WARMUP_FRACCION)\n    # nunca descartar argumentos en silencio.\n    descartados = sorted(k for k in params if k not in sig)\n    if descartados:\n        raise RuntimeError(f"TrainingArguments no acepta {descartados} en esta versión de transformers.")\n    return TrainingArguments(**params)', 'notebook_fuente': '04c_RoBERTuito.ipynb', 'sha256_notebook_fuente': '99eddf8739d034ce8e097f33e8074cf98a1fb0caec6cbce0e4a2dcf4673abcd5', 'codigo_flujo_sha256': '7565cd917834e14b3fb92a310a8e41de6d917e103f86af941bc02758a95fea26', 'metadatos': {'mx': {'modelo_base': 'pysentimiento/robertuito-base-uncased', 'variante': 'mx', 'seed': 42, 'config_id': 'P_l2_b16_e6_w1_b_d3_s1_r9_i2_c0_m0', 'configuracion': {'preprocessing': 'PYSENTIMIENTO', 'learning_rate': 2e-05, 'batch_size': 16, 'num_train_epochs': 6, 'weight_decay': 0.01, 'loss_mode': 'balanced', 'classifier_dropout': 0.3, 'label_smoothing': 0.1, 'lr_decay': 0.9, 'dropout_interno': 0.2, 'congelar_frac': 0.0, 'mascara_tokens': 0.0, 'refit_epochs': 6}, 'epocas_entrenadas': 6.0, 'n_train': 2399, 'csv_train': '..\\data\\transformers\\train_transformers_mx.csv', 'sha256_csv_train': '8728c2bfe70281f2bdc27b0993f7157e094ed6494d015518a1acbfb20f5f9d7d', 'columna_texto': 'MESSAGE_TRANSFORMER', 'max_length': 128, 'temperatura': 1.1835394289311814, 'etiquetas': {'0': 'NO_IRONICO', '1': 'IRONICO'}, 'versiones': {'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'optuna': '5.0.0', 'numpy': '1.26.4', 'pandas': '3.0.3'}, 'fecha': '2026-10-28T18:02:24'}, 'es': {'modelo_base': 'pysentimiento/robertuito-base-uncased', 'variante': 'es', 'seed': 42, 'config_id': 'P_l2_b16_e6_w1_b_d1_s1_r9_i2_c0_m0', 'configuracion': {'preprocessing': 'PYSENTIMIENTO', 'learning_rate': 2e-05, 'batch_size': 16, 'num_train_epochs': 6, 'weight_decay': 0.01, 'loss_mode': 'balanced', 'classifier_dropout': 0.1, 'label_smoothing': 0.1, 'lr_decay': 0.9, 'dropout_interno': 0.2, 'congelar_frac': 0.0, 'mascara_tokens': 0.0, 'refit_epochs': 5}, 'epocas_entrenadas': 5.0, 'n_train': 2397, 'csv_train': '..\\data\\transformers\\train_transformers_es.csv', 'sha256_csv_train': '4526a9bedeceb6cf9b0f95cb7f2023bb21fc661245c9d69d822be682902e6524', 'columna_texto': 'MESSAGE_TRANSFORMER', 'max_length': 128, 'temperatura': 1.1332381911864717, 'etiquetas': {'0': 'NO_IRONICO', '1': 'IRONICO'}, 'versiones': {'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'optuna': '5.0.0', 'numpy': '1.26.4', 'pandas': '3.0.3'}, 'fecha': '2026-10-28T18:03:06'}, 'cu': {'modelo_base': 'pysentimiento/robertuito-base-uncased', 'variante': 'cu', 'seed': 42, 'config_id': 'P_l2_b16_e6_w1_b_d1_s1_r9_i2_c0_m0', 'configuracion': {'preprocessing': 'PYSENTIMIENTO', 'learning_rate': 2e-05, 'batch_size': 16, 'num_train_epochs': 6, 'weight_decay': 0.01, 'loss_mode': 'balanced', 'classifier_dropout': 0.1, 'label_smoothing': 0.1, 'lr_decay': 0.9, 'dropout_interno': 0.2, 'congelar_frac': 0.0, 'mascara_tokens': 0.0, 'refit_epochs': 5}, 'epocas_entrenadas': 5.0, 'n_train': 2400, 'csv_train': '..\\data\\transformers\\train_transformers_cu.csv', 'sha256_csv_train': 'ab8a76cf77a4189ce5921d3edb6dc174d6b9c2e090ce7654f376ee92e3b1b05a', 'columna_texto': 'MESSAGE_TRANSFORMER', 'max_length': 128, 'temperatura': 1.1631564126448122, 'etiquetas': {'0': 'NO_IRONICO', '1': 'IRONICO'}, 'versiones': {'torch': '2.14.0+cu130', 'transformers': '5.17.0', 'optuna': '5.0.0', 'numpy': '1.26.4', 'pandas': '3.0.3'}, 'fecha': '2026-10-28T18:03:49'}}}}

FUENTES_FIGURAS = {'configuracion': 'import os\nimport re\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom IPython.display import display\n\n# Permite ejecutar desde la raíz del proyecto o desde notebooks/.\ncandidatos_data = [Path.cwd() / "data", Path.cwd().parent / "data"]\nDATA_DIR = next((p for p in candidatos_data if p.is_dir()), None)\nif DATA_DIR is None:\n    raise FileNotFoundError("Ejecute desde la raíz del proyecto o desde notebooks/.")\n# La variable de entorno permite verificar las figuras en una carpeta alternativa.\nOUT_DIR = Path(os.environ.get("SHAP_OUTPUT_DIR", str(DATA_DIR / "resultados_interpretabilidad")))\nOUT_DIR.mkdir(parents=True, exist_ok=True)\nVARIANTES = ["mx", "es", "cu"]\nTRANSFORMERS = {"BERT": "beto", "DistilBERT": "distilbeto", "RoBERTuito": "robertuito"}\nML_DIR = DATA_DIR / "resultados_ml_groupcv_tfidf_lr_rf_10k"\nTOP_N = 20  # palabras "más irónicas" por modelo para la coincidencia\nMODELO_PRINCIPAL = "RoBERTuito"\nML_PRINCIPAL = "LR"\n\n\ndef normalizar(texto):\n    t = unicodedata.normalize("NFKD", str(texto).lower().strip())\n    t = "".join(ch for ch in t if not unicodedata.combining(ch))\n    return t.lstrip("#@").strip()\n\n\nGLOBAL, TOKENS = [], []\nfor nombre, pref in TRANSFORMERS.items():\n    carpeta = DATA_DIR / f"resultados_{pref}_grouped_5x3_tpe20"\n    g = carpeta / f"{pref}_shap_global.csv"\n    # Contribuciones por palabra (suma de sus tokens); si no existen, por token.\n    t = carpeta / f"{pref}_shap_palabras.csv"\n    t = t if t.exists() else carpeta / f"{pref}_shap_tokens.csv"\n    if g.exists():\n        d = pd.read_csv(g)\n        d = d.rename(columns={"token_normalizado": "termino"}).assign(modelo=nombre, familia="Transformer")\n        GLOBAL.append(d[["variante", "modelo", "familia", "termino", "n_textos", "shap_medio", "shap_abs_medio"]])\n    else:\n        print(f"AVISO: falta {g} (ejecuta el notebook 04 de {nombre}).")\n    if t.exists():\n        TOKENS.append(pd.read_csv(t, dtype={"ID": str}).assign(modelo=nombre))\ng_ml = ML_DIR / "ml_shap_global.csv"\nif g_ml.exists():\n    d = pd.read_csv(g_ml).rename(columns={"palabra": "termino"}).assign(familia="ML")\n    GLOBAL.append(d[["variante", "modelo", "familia", "termino", "n_textos", "shap_medio", "shap_abs_medio"]])\nelse:\n    print(f"AVISO: falta {g_ml} (ejecuta el notebook 02, sección 11).")\nif not GLOBAL:\n    raise FileNotFoundError("No hay resultados de SHAP. Ejecuta primero los notebooks 02 y 04.")\nGLOBAL = pd.concat(GLOBAL, ignore_index=True)\nGLOBAL["termino_norm"] = GLOBAL["termino"].map(normalizar)\nGLOBAL = GLOBAL[GLOBAL["termino_norm"].str.len() > 1]\nTOKENS = pd.concat(TOKENS, ignore_index=True) if TOKENS else pd.DataFrame()\nMODELOS = [m for m in [*TRANSFORMERS, "LR", "RF", "XGBoost"] if (GLOBAL["modelo"] == m).any()]\nprint("Modelos con SHAP:", MODELOS)\n\n# El ensamble se explica como función completa de mayoría, no como promedio de componentes.\nVOTACION_SHAP_DIR = Path(os.environ.get("SHAP_VOTACION_DIR", str(DATA_DIR / "resultados_interpretabilidad/votacion_ml")))\nruta_votacion = VOTACION_SHAP_DIR / "shap_votacion_global.csv"\nif ruta_votacion.exists():\n    g_votacion = pd.read_csv(ruta_votacion)\n    g_votacion = g_votacion[g_votacion["tipo"] == "palabra"].assign(familia="ML")\n    g_votacion["termino_norm"] = g_votacion["termino"].map(normalizar)\n    g_votacion = g_votacion[g_votacion["termino_norm"].str.len() > 1]\n    GLOBAL = pd.concat([GLOBAL, g_votacion[["variante", "modelo", "familia", "termino",\n        "n_textos", "shap_medio", "shap_abs_medio", "termino_norm"]]], ignore_index=True)\n', 'global': '# Figuras para el artículo: solo se leen resultados SHAP ya calculados.\nimport html\nimport json\nfrom matplotlib.colors import to_hex\nfrom matplotlib.font_manager import FontProperties\nfrom matplotlib.ticker import MaxNLocator\nfrom matplotlib.patches import Patch\n\nARTICULO_DIR = OUT_DIR / "articulo"\nARTICULO_DIR.mkdir(parents=True, exist_ok=True)\nPAISES = {"mx": "México", "es": "España", "cu": "Cuba"}\nNOMBRES_MODELOS = {"LR": "Regresión logística", "RF": "Random Forest", "XGBoost": "XGBoost", "HardVoting": "Votación de ML"}\nCOLOR_POSITIVO, COLOR_NEGATIVO = "#b83c46", "#28689b"\nN_TERMINOS_DIRECCION = 5\nMIN_TEXTOS_GLOBAL = 5\nN_BARRAS_LOCAL = 8\n\n\ndef guardar_figura_articulo(fig, nombre):\n    for extension in ("png", "svg"):\n        fig.savefig(ARTICULO_DIR / f"{nombre}.{extension}", dpi=300,\n                    bbox_inches="tight", facecolor="white")\n\n\n# Mismo criterio que las matrices: ganadores del test filtrado, con ensambles candidatos.\nruta_test = DATA_DIR / "resultados_comparacion_final/tabla_resumen_test.csv"\nif not ruta_test.exists():\n    raise FileNotFoundError(f"Falta {ruta_test}")\ntest = pd.read_csv(ruta_test)\nREFERENTES_ML = {}\nfor variante in VARIANTES:\n    candidatos = test[(test["variante"] == variante) &\n        test["modelo"].isin(["LR", "RF", "XGBoost", "HardVoting"])].dropna(subset=["f1_macro_mean"])\n    REFERENTES_ML[variante] = candidatos.sort_values(["f1_macro_mean", "modelo"],\n        ascending=[False, True]).iloc[0]["modelo"]\nif any(m == "HardVoting" for m in REFERENTES_ML.values()) and not ruta_votacion.exists():\n    raise FileNotFoundError("Falta SHAP de la votación. Ejecutar primero 07a_SHAP_votacion_ML.ipynb.")\npd.DataFrame([{"variante": v, "modelo": m, "criterio": "F1-Macro medio del test filtrado"}\n    for v,m in REFERENTES_ML.items()]).to_csv(ARTICULO_DIR / "referentes_ml.csv", index=False)\n\nfilas_globales = []\nwith plt.rc_context({"font.size": 10, "svg.fonttype": "none"}):\n    fig, axes = plt.subplots(2, len(VARIANTES), figsize=(15, 10), squeeze=False)\n    for j, variante in enumerate(VARIANTES):\n        for i, modelo in enumerate([MODELO_PRINCIPAL, REFERENTES_ML[variante]]):\n            ax = axes[i, j]\n            g = GLOBAL[(GLOBAL["variante"] == variante) & (GLOBAL["modelo"] == modelo) &\n                       (GLOBAL["n_textos"] >= MIN_TEXTOS_GLOBAL)].dropna(subset=["shap_medio"])\n            elegidos = pd.concat([\n                g[g["shap_medio"] > 0].nlargest(N_TERMINOS_DIRECCION, "shap_medio"),\n                g[g["shap_medio"] < 0].nsmallest(N_TERMINOS_DIRECCION, "shap_medio"),\n            ]).drop_duplicates("termino").sort_values("shap_medio")\n            if elegidos.empty:\n                ax.text(.5, .5, "Sin términos suficientes", ha="center", transform=ax.transAxes)\n                ax.set_axis_off()\n                continue\n            ax.barh(range(len(elegidos)), elegidos["shap_medio"],\n                    color=np.where(elegidos["shap_medio"] > 0, COLOR_POSITIVO, COLOR_NEGATIVO))\n            ax.set_yticks(range(len(elegidos)),\n                          [f"{t} · n={int(n)}" for t, n in\n                           zip(elegidos["termino"], elegidos["n_textos"])], fontsize=9)\n            ax.axvline(0, color="#444444", lw=.8)\n            ax.grid(axis="x", alpha=.18)\n            ax.set_axisbelow(True)\n            ax.set_title(f"{PAISES[variante]} · {NOMBRES_MODELOS.get(modelo, modelo)}", pad=9)\n            ax.spines[["top", "right"]].set_visible(False)\n            filas_globales.extend(elegidos.assign(panel_fila=i + 1).to_dict("records"))\n    # Mismos límites y marcas en todos los paneles de un mismo modelo.\n    # RoBERTuito explica probabilidades, LR log-odds y la votación una decisión binaria.\n    ESCALAS_GLOBALES = {}\n    UNIDADES_SHAP = {"RoBERTuito": "probabilidad", "LR": "log-odds", "RF": "probabilidad", "HardVoting": "salida binaria"}\n    for modelo, valores_modelo in pd.DataFrame(filas_globales).groupby("modelo"):\n        maximo = max(float(valores_modelo["shap_medio"].abs().max()), 1e-8)\n        marcas = MaxNLocator(nbins=5, symmetric=True, min_n_ticks=3).tick_values(-1.08 * maximo, 1.08 * maximo)\n        ESCALAS_GLOBALES[modelo] = {"limites": (float(marcas[0]), float(marcas[-1])),\n                                    "marcas": marcas.tolist(), "unidad": UNIDADES_SHAP[modelo]}\n    for j, variante in enumerate(VARIANTES):\n        for i, modelo in enumerate([MODELO_PRINCIPAL, REFERENTES_ML[variante]]):\n            ax = axes[i, j]\n            if modelo not in ESCALAS_GLOBALES or not ax.axison:\n                continue\n            escala = ESCALAS_GLOBALES[modelo]\n            ax.set_xlim(*escala["limites"])\n            ax.set_xticks(escala["marcas"])\n    EJES_GLOBALES = axes  # Permite verificar las escalas sin recalcular SHAP.\n    fig.suptitle("Términos con contribuciones hacia ironía y no ironía", fontsize=15, y=1.01)\n    # Cada panel identifica su eje para evitar que el rótulo parezca exclusivo de España.\n    for ax in axes.flat:\n        ax.set_xlabel("Contribución SHAP media", fontsize=10, labelpad=6)\n    fig.legend(handles=[Patch(facecolor=COLOR_POSITIVO, label="Hacia ironía"),\n                        Patch(facecolor=COLOR_NEGATIVO, label="Hacia no ironía")],\n               loc="lower center", bbox_to_anchor=(.5, -.004), ncol=2,\n               frameon=False, fontsize=10)\n    fig.tight_layout(h_pad=3, w_pad=2, rect=[0, .05, 1, 1])\n    guardar_figura_articulo(fig, "shap_global_articulo")\n    display(fig)\n    plt.close(fig)\npd.DataFrame(filas_globales).to_csv(ARTICULO_DIR / "shap_global_articulo.csv", index=False)\nprint("Referentes ML:", REFERENTES_ML)\n', 'local': '# Casos ilustrativos del ganador del test en las tres variantes.\nEJEMPLOS_ARTICULO = [\n    {"variante": "mx", "modelo": "RoBERTuito", "tipo": "FP", "ID": "7848dd4a3d97f9e1ec61b927789ebf18"},\n    {"variante": "es", "modelo": "RoBERTuito", "tipo": "VP", "ID": "987610faf6fde801af91fb59f62f38c9"},\n    {"variante": "cu", "modelo": "RoBERTuito", "tipo": "FP", "ID": "9a95b2dc61f5a657f2794e2d4995e22d"},\n]\nTIPOS_ETIQUETAS = {"VP": (1, 1), "VN": (0, 0), "FP": (0, 1), "FN": (1, 0)}\nETIQUETAS_TEXTO = {0: "no irónico", 1: "irónico"}\nTIPOS_TEXTO = {"VP": "Verdadero positivo", "VN": "Verdadero negativo",\n               "FP": "Falso positivo", "FN": "Falso negativo"}\n\n\ndef cargar_ejemplo_articulo(especificacion):\n    modelo, variante, identificador = (especificacion[k] for k in ("modelo", "variante", "ID"))\n    prefijo = TRANSFORMERS.get(modelo)\n    carpeta = DATA_DIR / f"resultados_{prefijo}_grouped_5x3_tpe20" if prefijo else ML_DIR\n    # El archivo completo permite elegir cualquier ID con SHAP guardado, no solo los nueve ejemplos previos.\n    ruta = (carpeta / f"{prefijo}_test_predicciones_modelo_exportado.csv" if prefijo\n            else carpeta / "ml_shap_ejemplos.csv")\n    ejemplos = pd.read_csv(ruta, dtype={"ID": str})\n    seleccion = ejemplos[(ejemplos["variante"] == variante) & (ejemplos["ID"] == identificador)]\n    if not prefijo:\n        seleccion = seleccion[seleccion["modelo"] == modelo]\n    if len(seleccion) != 1:\n        raise ValueError(f"Se esperaba un registro para {modelo}/{variante}/{identificador}.")\n    fila = seleccion.iloc[0]\n    if fila["tipo"] != especificacion["tipo"]:\n        raise ValueError("El tipo de resultado indicado no coincide con el archivo guardado.")\n    real, predicha = TIPOS_ETIQUETAS[fila["tipo"]]\n    fragmentos = []\n    if prefijo:\n        if int(fila["y_true"]) != real or int(fila["y_pred"]) != predicha:\n            raise ValueError("Las etiquetas no corresponden al tipo de resultado.")\n        if str(fila["coincide_train"]).lower() != "false":\n            raise ValueError("El caso coincide con el entrenamiento según el filtro del modelo.")\n        palabras = pd.read_csv(carpeta / f"{prefijo}_shap_palabras.csv", dtype={"ID": str})\n        palabras = palabras[(palabras["variante"] == variante) &\n                            (palabras["ID"] == identificador)].sort_values("posicion")\n        if palabras.empty or palabras["posicion"].duplicated().any():\n            raise ValueError("Las contribuciones están vacías o tienen posiciones repetidas.")\n        if not (palabras["y_true"].eq(real).all() and palabras["y_pred"].eq(predicha).all()\n                and palabras["tipo"].eq(especificacion["tipo"]).all()):\n            raise ValueError("Las etiquetas no coinciden con sus contribuciones.")\n        if not np.allclose(palabras["prob_ironia"], float(fila["prob_ironia"]), atol=1e-7, rtol=0):\n            raise ValueError("Las probabilidades no coinciden con sus contribuciones.")\n        contribuciones = [(str(r.token), float(r.shap)) for r in palabras.itertuples()]\n        fragmentos = contribuciones  # Texto completo; no se limita el número de palabras.\n        explicado = str(fila["texto"])\n        if " ".join(t for t, _ in fragmentos).split() != explicado.split():\n            raise ValueError("Las palabras guardadas no reproducen el texto explicado completo.")\n    else:\n        contribuciones = []\n        for elemento in str(fila["contribuciones"]).split("; "):\n            termino, valor = elemento.rsplit("=", 1)\n            contribuciones.append((termino, float(valor)))\n        explicado = ""\n    if not contribuciones or not np.isfinite([v for _, v in contribuciones]).all():\n        raise ValueError("No hay contribuciones locales finitas para el caso seleccionado.")\n    return {**especificacion, "real": real, "predicha": predicha,\n            "prob_ironia": float(fila["prob_ironia"]),\n            "original": str(fila["mensaje_original"]), "explicado": explicado,\n            "contribuciones": contribuciones, "fragmentos": fragmentos}\n\n\nCASOS_ARTICULO = [cargar_ejemplo_articulo(e) for e in EJEMPLOS_ARTICULO]\nif (len(CASOS_ARTICULO) != 3 or\n        [c["variante"] for c in CASOS_ARTICULO] != ["mx", "es", "cu"] or\n        any(c["modelo"] != "RoBERTuito" or not c["fragmentos"] for c in CASOS_ARTICULO)):\n    raise ValueError("Se requiere una explicación guardada de RoBERTuito por variante.")\n\n# Hasta diez contribuciones por magnitud absoluta; presentación en el orden del texto.\nN_CONTRIBUCIONES_LOCALES = 10\nSELECCION_LOCAL = {}\nfor caso in CASOS_ARTICULO:\n    mayores = sorted(enumerate(caso["fragmentos"]), key=lambda item: (-abs(item[1][1]), item[0]))\n    SELECCION_LOCAL[caso["variante"]] = sorted(mayores[:N_CONTRIBUCIONES_LOCALES], key=lambda item: item[0])\nLIMITES_EJE_LOCAL = (-.12, .12)\nMARCAS_EJE_LOCAL = [-.1, 0., .1]\nif any(abs(v) >= LIMITES_EJE_LOCAL[1] for c in CASOS_ARTICULO for _, v in c["fragmentos"]):\n    raise ValueError("Una contribución excede el rango común; revise los límites antes de generar la figura.")\n\nALTURA_FILA_PT = 10.8\nwith plt.rc_context({"font.size": 8.5, "svg.fonttype": "none"}):\n    fig = plt.figure(figsize=(6.5, 3.2))\n    EJES_LOCALES, VALORES_ANOTADOS = [], []\n    for j, (letra, caso) in enumerate(zip("abc", CASOS_ARTICULO)):\n        centro = (j + .5) / 3\n        elegidos = SELECCION_LOCAL[caso["variante"]]\n        fig.text(centro, .98, f"{letra}. {PAISES[caso[\'variante\']]} · RoBERTuito",\n                 ha="center", va="top", fontsize=9, fontweight="bold")\n        fig.text(centro, .91, TIPOS_TEXTO[caso["tipo"]], ha="center", va="top", fontsize=8.5)\n        fig.text(centro, .85, f"P(irónico): {caso[\'prob_ironia\']:.3f}",\n                 ha="center", va="top", fontsize=8)\n        fig.text(centro, .79, f"{len(elegidos)} de {len(caso[\'fragmentos\'])} contribuciones",\n                 ha="center", va="top", fontsize=8, color="#444444")\n        altura = len(elegidos) * ALTURA_FILA_PT / (fig.get_size_inches()[1] * 72)\n        ax = fig.add_axes([j / 3 + .138, .70 - altura, .19, altura])\n        valores = [v for _, (_, v) in elegidos]\n        ax.barh(range(len(elegidos)), valores, height=.8,\n                 color=[COLOR_POSITIVO if v > 0 else COLOR_NEGATIVO for v in valores])\n        ax.set_yticks(range(len(elegidos)), [t for _, (t, _) in elegidos], fontsize=8.5)\n        ax.set_ylim(len(elegidos) - .5, -.5)\n        ax.axvline(0, color="#555555", lw=.75)\n        ax.set_xlim(*LIMITES_EJE_LOCAL)\n        ax.set_xticks(MARCAS_EJE_LOCAL)\n        ax.tick_params(axis="x", labelsize=8, length=3)\n        ax.tick_params(axis="y", length=2.5, pad=3)\n        ax.set_xlabel("Valor SHAP", fontsize=8.5, labelpad=3)\n        ax.spines[["top", "right"]].set_visible(False)\n        ax.grid(axis="x", alpha=.18)\n        ax.set_axisbelow(True)\n        for k, valor in enumerate(valores):\n            etiqueta = ax.text(valor + (.008 if valor >= 0 else -.008), k, f"{valor:.3f}",\n                               ha="left" if valor >= 0 else "right", va="center", fontsize=8)\n            VALORES_ANOTADOS.append((ax, etiqueta, valor))\n        EJES_LOCALES.append(ax)\n    # Si una etiqueta larga excede el eje, se coloca dentro de su barra.\n    fig.canvas.draw()\n    renderer = fig.canvas.get_renderer()\n    ETIQUETAS_INTERIORES = []\n    for eje, etiqueta, valor in VALORES_ANOTADOS:\n        borde = eje.get_window_extent(renderer)\n        caja = etiqueta.get_window_extent(renderer)\n        if caja.x0 < borde.x0 - 1 or caja.x1 > borde.x1 + 1:\n            ancho_barra = abs(eje.transData.transform((valor, 0))[0] - eje.transData.transform((0, 0))[0])\n            if caja.width + 4 <= ancho_barra:\n                etiqueta.set_x(valor / 2)\n                etiqueta.set_ha("center")\n                etiqueta.set_color("white")\n            else:\n                # En una barra corta el texto blanco quedaría parcialmente invisible.\n                etiqueta.set_x(-.008 if valor >= 0 else .008)\n                etiqueta.set_ha("right" if valor >= 0 else "left")\n                etiqueta.set_color("black")\n            ETIQUETAS_INTERIORES.append((eje, etiqueta, valor))\n    fig.legend(handles=[Patch(facecolor=COLOR_POSITIVO, label="Hacia ironía"),\n                        Patch(facecolor=COLOR_NEGATIVO, label="Hacia no ironía")],\n               loc="lower center", bbox_to_anchor=(.5, -.005), ncol=2,\n               frameon=False, fontsize=8.5)\n    guardar_figura_articulo(fig, "shap_local_articulo")\n    display(fig)\n    plt.close(fig)\n\n# Se conservan todas las contribuciones y se identifican las mostradas en barras.\nresumen_casos, filas_contribuciones, detalles, transcripciones, tarjetas_texto = [], [], [], [], []\nfor letra, caso in zip("abc", CASOS_ARTICULO):\n    resumen_casos.append({k: v for k, v in caso.items() if k not in ("contribuciones", "fragmentos")})\n    posiciones = {p for p, _ in SELECCION_LOCAL[caso["variante"]]}\n    tabla = []\n    for posicion, (termino, valor) in enumerate(caso["contribuciones"]):\n        seleccionada = posicion in posiciones\n        filas_contribuciones.append({"variante": caso["variante"], "modelo": caso["modelo"],\n                                     "ID": caso["ID"], "tipo": caso["tipo"], "posicion": posicion,\n                                     "termino": termino, "shap": valor,\n                                     "mostrada_en_barras": seleccionada, "mostrada_en_texto": True})\n        tabla.append(f\'<tr><td>{posicion + 1}</td><td>{html.escape(termino)}</td><td>{valor:+.6f}</td>\'\n                     f\'<td>{"Sí" if seleccionada else "No"}</td></tr>\')\n    texto_presentado = " ".join(caso["original"].replace("\\\\n", "\\n").split())\n    transcripciones.append(f"**{letra}. {PAISES[caso[\'variante\']]}:** «{texto_presentado}»")\n    tarjetas_texto.append(f\'<p><b>{letra}. {PAISES[caso["variante"]]}:</b> «{html.escape(texto_presentado)}»</p>\')\n    detalles.append(f\'<details><summary>{PAISES[caso["variante"]]}: todas las contribuciones</summary>\'\n                    \'<table><thead><tr><th>Posición</th><th>Palabra</th><th>SHAP</th><th>En barras</th></tr></thead>\'\n                    f\'<tbody>{"".join(tabla)}</tbody></table></details>\')\nTEXTOS_LOCALES_ARTICULO = \'\\n\\n\'.join(transcripciones)\n(ARTICULO_DIR / \'shap_local_textos_articulo.md\').write_text(TEXTOS_LOCALES_ARTICULO + \'\\n\', encoding=\'utf-8\')\npd.DataFrame(resumen_casos).to_csv(ARTICULO_DIR / "shap_local_casos.csv", index=False)\npd.DataFrame(filas_contribuciones).to_csv(ARTICULO_DIR / "shap_local_contribuciones.csv", index=False)\npagina = (\'<!doctype html><html lang="es"><meta charset="utf-8"><title>RoBERTuito: explicaciones locales SHAP</title>\'\n          \'<style>body{font:16px/1.5 system-ui;max-width:1050px;margin:25px auto;padding:0 20px;color:#222}\'\n          \'img{width:100%;height:auto}details{border-top:1px solid #ccc;padding:12px 0}\'\n          \'td,th{padding:3px 20px 3px 0;text-align:left}table{border-collapse:collapse}</style>\'\n          \'<h1>RoBERTuito: tres variantes, tres casos ilustrativos</h1>\'\n          \'<img src="shap_local_articulo.svg" alt="Tres paneles SHAP de RoBERTuito en una fila, hasta diez contribuciones por caso y eje común">\'\n          \'<p>Las barras muestran hasta diez contribuciones de mayor magnitud absoluta por caso, en orden de aparición. Las palabras omitidas no tienen necesariamente contribución nula. \'\n          \'Los valores están en la escala de probabilidad; las barras no incluyen el valor de referencia. \'\n          \'Los ejes horizontales son comunes.</p>\'\n          \'<h2>Textos de los paneles</h2>\' + \'\'.join(tarjetas_texto) + \'\'.join(detalles) + \'</html>\')\n(ARTICULO_DIR / "shap_local_articulo.html").write_text(pagina, encoding="utf-8")\nprint("Figura compacta de tres paneles, textos externos, HTML y CSV guardados en:", ARTICULO_DIR.resolve())\n'}

## 2. Funciones del flujo final

Se define el procedimiento de preparación, entrenamiento, inferencia, métricas y SHAP.

In [ ]:
"""Prepara una ejecución final separada, con configuraciones ya seleccionadas."""
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import io
import json
import os
import shutil

import numpy as np
import pandas as pd

SEMILLAS_FINALES_FP32 = [42,52,62,72,82]
VARIANTES_FP32 = ['mx','es','cu']
TAMANO_TRAIN_FP32 = {'mx':2399,'es':2397,'cu':2400}
TAMANO_TEST_FP32 = {'mx':598,'es':577,'cu':599}


def sha_archivo_fp32(ruta):
    h=hashlib.sha256()
    with Path(ruta).open('rb') as f:
        for bloque in iter(lambda:f.read(8*1024*1024),b''):h.update(bloque)
    return h.hexdigest()


def escribir_json_fp32(objeto,ruta):
    ruta=Path(ruta);ruta.parent.mkdir(parents=True,exist_ok=True)
    tmp=ruta.with_name(ruta.name+'.tmp')
    tmp.write_text(json.dumps(objeto,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    tmp.replace(ruta)


def escribir_csv_fp32(df,ruta):
    ruta=Path(ruta);ruta.parent.mkdir(parents=True,exist_ok=True)
    tmp=ruta.with_name(ruta.name+'.tmp')
    df.to_csv(tmp,index=False);tmp.replace(ruta)


def compatibilidad_csv_fp32(ruta,esperado):
    """Solo acepta el archivo original o una variante verificable de saltos de línea."""
    bruto=Path(ruta).read_bytes()
    df=pd.read_csv(io.BytesIO(bruto),dtype={'ID':str})
    if hashlib.sha256(bruto).hexdigest()==esperado:
        return 'exacto',df
    # El checkout de Git en macOS puede normalizar los saltos de registro y de campos.
    # Reconstruir el hash evita aceptar cambios de contenido por un simple recuento.
    for finales in ['\n','\r\n']:
        for internos in [False,True]:
            candidato=df.copy()
            if internos:
                for c in candidato.columns:
                    if pd.api.types.is_string_dtype(candidato[c]):
                        candidato[c]=candidato[c].str.replace('\r\n','\n',regex=False).str.replace('\n','\r\n',regex=False)
            serializado=candidato.to_csv(index=False,lineterminator=finales).encode('utf-8')
            if hashlib.sha256(serializado).hexdigest()==esperado:
                return 'equivalencia_saltos_de_linea',candidato
    raise ValueError(f'{ruta}: no coincide con el archivo de entrenamiento registrado. Se detiene antes de entrenar.')


def preparar_final_fp32(raiz,salida,fuentes):
    raiz=Path(raiz).resolve();salida=Path(salida).resolve();datos=raiz/'data'
    if salida==datos or salida in [datos.parent,raiz]:
        raise ValueError('La salida debe ser una carpeta nueva dentro de data, separada de los resultados anteriores.')
    if salida.exists() and any(salida.iterdir()) and not (salida/'manifiesto_fp32.json').exists():
        raise ValueError('La carpeta de salida contiene archivos ajenos a este flujo. Elige un nombre nuevo.')
    entradas={};plan=[];pares_modelos={};hashes={}
    def registrar(p):hashes[str(p.relative_to(raiz))]=sha_archivo_fp32(p)
    comunes=datos/'resultados_comparacion_final/ids_test_coincidentes_con_train.csv'
    excl=pd.read_csv(comunes,dtype={'ID':str});registrar(comunes)
    ids_comunes={(r.variante,r.ID) for r in excl.itertuples()}
    for v in VARIANTES_FP32:
        assert 600-sum(a==v for a,_ in ids_comunes)==TAMANO_TEST_FP32[v]
        tr=datos/'transformers'/f'train_transformers_{v}.csv'
        te=datos/'transformers'/f'test_transformers_{v}.csv'
        train=pd.read_csv(tr,dtype={'ID':str});test=pd.read_csv(te,dtype={'ID':str})
        assert len(train)==TAMANO_TRAIN_FP32[v] and len(test)==600
        assert not train.ID.duplicated().any() and not test.ID.duplicated().any()
        assert set(train.IS_IRONIC)==set(test.IS_IRONIC)=={0,1}
        assert not train[['MESSAGE_TRANSFORMER','IS_IRONIC']].isna().any().any()
        assert not test[['MESSAGE_TRANSFORMER','IS_IRONIC']].isna().any().any()
        registrar(tr);registrar(te)
    for modelo,f in fuentes.items():
        carpeta=datos/f['carpeta']
        for v in VARIANTES_FP32:
            src=carpeta/'modelo_final'/v
            meta=json.loads((src/'metadatos.json').read_text(encoding='utf-8'))
            esperado=f['metadatos'][v]
            assert meta['seed']==42 and meta['configuracion']==esperado['configuracion']
            assert meta['config_id']==esperado['config_id']
            cfg=meta['configuracion']
            assert cfg['learning_rate']==2e-5 and cfg['batch_size']==16 and cfg['num_train_epochs']==6
            assert cfg['label_smoothing']==.1 and cfg['lr_decay']==.9 and cfg['dropout_interno']==.2
            assert cfg['congelar_frac']==0 and cfg['refit_epochs'] in [5,6]
            assert cfg['mascara_tokens']==(0 if modelo=='RoBERTuito' else .15)
            compat,original_compatible=compatibilidad_csv_fp32(datos/'transformers'/f'train_transformers_{v}.csv',meta['sha256_csv_train'])
            entradas[(modelo,v)]={'meta':meta,'compatibilidad_csv':compat}
            for p in [src/'model.safetensors',src/'metadatos.json',src/'config.json',
                      src/'tokenizer.json',src/'tokenizer_config.json']:
                registrar(p)
            pred=carpeta/f'{f["prefijo"]}_test_probabilidades.csv'
            ex=carpeta/f'{f["prefijo"]}_test_predicciones_modelo_exportado.csv'
            registrar(pred);registrar(ex)
            pares_modelos[(modelo,v)]=src
            for seed in SEMILLAS_FINALES_FP32:
                plan.append({'modelo':modelo,'variante':v,'seed':seed,
                    'accion':'reutilizar_pesos_originales' if seed==42 else 'entrenar_configuracion_final',
                    'epochs':int(cfg['refit_epochs']),
                    'directorio':str(Path(f['carpeta'])/'modelos_por_semilla'/f'seed{seed}'/v)})
    # Se conservan entradas y resultados anteriores; el manifiesto permite comprobarlo al finalizar.
    for nombre in ['predicciones_test_ml.csv','seleccion_final_ml.csv']:
        registrar(datos/'resultados_ml_groupcv_tfidf_lr_rf_10k'/nombre)
    registrar(datos/'resultados_ml_hard_voting_tfidf10k/hard_voting_test_predicciones.csv')
    for nombre in ['tabla_resumen_test.csv','metricas_test_por_semilla.csv']:
        registrar(datos/'resultados_comparacion_final'/nombre)
    protocolo={'semillas':SEMILLAS_FINALES_FP32,'max_length':128,'batch_inferencia':16,
        'precision_entrenamiento':'BF16, como el protocolo anterior',
        'precision_inferencia':'FP32, autocast desactivado, matmul highest',
        'seleccion':'configuraciones y epochs ya elegidos; sin búsquedas ni selección por test',
        'shap':{'algoritmo':'partition','max_evals':500,'n_por_tipo':50,'semilla':42,'filtro':'propio'},
        'fuentes_sha256':{m:hashlib.sha256(f['definiciones'].encode()).hexdigest() for m,f in fuentes.items()},
        'modelos':{m:{v:f['metadatos'][v]['configuracion'] for v in VARIANTES_FP32} for m,f in fuentes.items()},
        'archivos_originales_sha256':hashes}
    ruta_man=salida/'manifiesto_fp32.json'
    if ruta_man.exists():
        previo=json.loads(ruta_man.read_text(encoding='utf-8'))
        assert previo['protocolo']==protocolo,'Cambió una entrada o el protocolo; no se mezclan ejecuciones.'
    else:
        escribir_json_fp32({'creado_utc':datetime.now(timezone.utc).isoformat(),'protocolo':protocolo,
            'codigo_flujo_inicial_sha256':next(iter(fuentes.values()))['codigo_flujo_sha256']},ruta_man)
    escribir_json_fp32({'codigo_flujo_sha256':next(iter(fuentes.values()))['codigo_flujo_sha256'],
        'fecha_utc':datetime.now(timezone.utc).isoformat()},salida/'auditoria_codigo_ultima_ejecucion.json')
    escribir_csv_fp32(pd.DataFrame(plan),salida/'plan_45_modelos.csv')
    filas=[{'modelo':m,'variante':v,'compatibilidad_csv_original':d['compatibilidad_csv']} for (m,v),d in entradas.items()]
    escribir_csv_fp32(pd.DataFrame(filas),salida/'auditoria_datos.csv')
    print('Plan: 9 modelos reutilizados y 36 entrenamientos finales. Sin Optuna ni nested CV.')
    print('Salida separada:',salida)
    return {'raiz':raiz,'datos':datos,'salida':salida,'fuentes':fuentes,'plan':plan,'entradas':entradas,
            'originales':hashes,'ids_comunes':ids_comunes}


def verificar_originales_fp32(ctx):
    for nombre,h in ctx['originales'].items():
        assert sha_archivo_fp32(ctx['raiz']/nombre)==h,f'Cambió el archivo original {nombre}'
    print('Archivos originales sin cambios.')


"""Entrenamientos finales pendientes y una única función de inferencia FP32."""
import gc
import importlib.metadata
import inspect
import random
import re
import time
from contextlib import redirect_stdout,redirect_stderr


def ajustar_inferencia_fp32():
    import torch
    torch.set_float32_matmul_precision('highest')


def probabilidades_fp32(modelo,tokenizador,textos,lote=16):
    import torch
    ajustar_inferencia_fp32()
    modelo=modelo.float().eval()
    dispositivo=next(modelo.parameters()).device
    salidas=[]
    for i in range(0,len(textos),lote):
        enc=tokenizador([str(t) for t in textos[i:i+lote]],truncation=True,max_length=128,
            padding=True,pad_to_multiple_of=8,return_tensors='pt').to(dispositivo)
        with torch.inference_mode(),torch.autocast(device_type=dispositivo.type,enabled=False):
            salidas.append(modelo(**enc).logits.float().cpu().numpy())
    logits=np.vstack(salidas).astype(np.float64)
    z=logits-logits.max(axis=1,keepdims=True)
    prob=np.exp(z);prob/=prob.sum(axis=1,keepdims=True)
    assert np.isfinite(prob).all() and np.allclose(prob.sum(axis=1),1)
    return prob


def entorno_gpu_fp32(ctx):
    import torch,transformers,accelerate
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('Ejecuta este cuaderno en la GPU CUDA con BF16 del entrenamiento original.')
    versiones={k:importlib.metadata.version(k) for k in ['torch','transformers','numpy','pandas','accelerate','scikit-learn','safetensors']}
    for dato in ctx['entradas'].values():
        for k in ['torch','transformers','numpy']:
            assert versiones[k]==dato['meta']['versiones'][k],f'Usa el entorno original: {k} cambió de versión.'
    escribir_json_fp32({'versiones':versiones,'gpu':torch.cuda.get_device_name(),
        'cuda':torch.version.cuda,'precision_inferencia':'FP32, autocast=False, matmul=highest'},ctx['salida']/'entorno.json')
    ctx['versiones']=versiones
    ctx['protocolo_id']=hashlib.sha256(json.dumps(json.loads((ctx['salida']/'manifiesto_fp32.json').read_text())['protocolo'],sort_keys=True).encode()).hexdigest()
    return versiones


def comprobar_datos_tokenizados_fp32(ctx,modelo,variante,tok):
    meta=ctx['entradas'][(modelo,variante)]['meta']
    trainpath=ctx['datos']/'transformers'/f'train_transformers_{variante}.csv'
    train=pd.read_csv(trainpath,dtype={'ID':str})
    compat,antiguo=compatibilidad_csv_fp32(trainpath,meta['sha256_csv_train'])
    assert train.ID.tolist()==antiguo.ID.tolist()
    np.testing.assert_array_equal(train.IS_IRONIC,antiguo.IS_IRONIC)
    if compat!='exacto':
        # La equivalencia del CSV por saltos de línea debe ser también equivalencia
        # de entradas del modelo, no solo de textos para un lector humano.
        a=tok(train.MESSAGE_TRANSFORMER.tolist(),truncation=True,max_length=128,padding=False)['input_ids']
        b=tok(antiguo.MESSAGE_TRANSFORMER.tolist(),truncation=True,max_length=128,padding=False)['input_ids']
        assert a==b,'Los cambios de saltos de línea cambian tokens. Recupera el archivo original antes de entrenar.'
    return train


def contexto_entrenamiento_fp32(ctx,modelo,tok,run_dir,base):
    import torch
    import torch.nn.functional as F
    from transformers import AutoConfig,AutoModelForSequenceClassification,Trainer,TrainerCallback,TrainingArguments,set_seed
    from sklearn.utils.class_weight import compute_class_weight
    from sklearn.metrics import f1_score,accuracy_score,precision_score,recall_score
    ns=dict(np=np,torch=torch,F=F,random=random,gc=gc,re=re,inspect=inspect,
        redirect_stdout=redirect_stdout,redirect_stderr=redirect_stderr,
        AutoConfig=AutoConfig,AutoModelForSequenceClassification=AutoModelForSequenceClassification,
        Trainer=Trainer,TrainerCallback=TrainerCallback,TrainingArguments=TrainingArguments,set_seed=set_seed,
        compute_class_weight=compute_class_weight,f1_score=f1_score,accuracy_score=accuracy_score,
        precision_score=precision_score,recall_score=recall_score,Dataset=torch.utils.data.Dataset,
        tokenizer=tok,MODEL_NAME=str(base),RUN_DIR=run_dir,MAX_LENGTH=128,
        WARMUP_FRACCION=0.,USE_BF16=True,USE_FP16=False,PATRON_CAPA=re.compile(r'\.layer\.(\d+)\.'))
    run_dir.mkdir(parents=True,exist_ok=True)
    exec(compile(ctx['fuentes'][modelo]['definiciones'],f'funciones_originales_{modelo}','exec'),ns)
    return ns


def comprobar_modelo_completo_fp32(ctx,destino,modelo,variante,seed):
    ruta=destino/'completo_fp32.json'
    if not ruta.exists():return False
    j=json.loads(ruta.read_text(encoding='utf-8'))
    assert j['protocolo_id']==ctx['protocolo_id'] and j['seed']==seed
    assert j['modelo']==modelo and j['variante']==variante
    assert sha_archivo_fp32(destino/'model.safetensors')==j['sha256_pesos']
    return True


def obtener_modelo_final_fp32(ctx,modelo,v,seed,tok,train,bases):
    import torch
    from transformers import AutoModelForSequenceClassification,DataCollatorWithPadding
    from transformers.trainer_utils import get_last_checkpoint
    from huggingface_hub import snapshot_download
    f=ctx['fuentes'][modelo];meta=ctx['entradas'][(modelo,v)]['meta'];cfg=meta['configuracion']
    carpeta=ctx['salida']/f['carpeta'];destino=carpeta/'modelos_por_semilla'/f'seed{seed}'/v
    if comprobar_modelo_completo_fp32(ctx,destino,modelo,v,seed):
        print(f'{modelo} {v} semilla {seed}: modelo final ya guardado.',flush=True);return destino
    destino.mkdir(parents=True,exist_ok=True)
    progreso=destino/'exportacion_en_progreso.json'
    if progreso.exists():
        assert json.loads(progreso.read_text())['protocolo_id']==ctx['protocolo_id']
    elif any(destino.iterdir()):
        raise ValueError(f'{destino} contiene archivos sin procedencia de este flujo. No se sobrescriben.')
    escribir_json_fp32({'protocolo_id':ctx['protocolo_id'],'modelo':modelo,'variante':v,'seed':seed},progreso)
    t0=time.monotonic();origen='reutilizado'
    if seed==42:
        src=ctx['datos']/f['carpeta']/'modelo_final'/v
        shutil.copytree(src,destino,dirs_exist_ok=True)
        assert sha_archivo_fp32(src/'model.safetensors')==sha_archivo_fp32(destino/'model.safetensors')
    else:
        origen='nuevo_entrenamiento_final'
        if modelo not in bases:
            # Fijar una revisión concreta para todas las semillas nuevas de esta ejecución.
            pin=ctx['salida']/'bases'/f'{f["prefijo"]}.json'
            previo=json.loads(pin.read_text()) if pin.exists() else {}
            revision=previo.get('revision')
            try:
                base=Path(snapshot_download(repo_id=meta['modelo_base'],revision=revision,local_files_only=True))
            except Exception as exc:
                from huggingface_hub.errors import LocalEntryNotFoundError
                if not isinstance(exc,LocalEntryNotFoundError):raise
                base=Path(snapshot_download(repo_id=meta['modelo_base'],revision=revision,
                    allow_patterns=['config.json','model*.safetensors','model*.safetensors.index.json']))
            assert (base/'config.json').exists()
            if not list(base.glob('*.safetensors')) and not list(base.glob('pytorch_model*.bin')):
                base=Path(snapshot_download(repo_id=meta['modelo_base'],revision=base.name,
                    allow_patterns=['config.json','pytorch_model*.bin','pytorch_model*.bin.index.json']))
            if revision:assert base.name==revision
            escribir_json_fp32({'repositorio':meta['modelo_base'],'revision':base.name,'ruta_cache':str(base)},pin)
            bases[modelo]=base
        run=carpeta/'trabajos'/v/f'seed{seed}'
        ckpt=run/'ckpt';ckpt.mkdir(parents=True,exist_ok=True)
        ns=contexto_entrenamiento_fp32(ctx,modelo,tok,run,bases[modelo])
        ns['establecer_semilla'](seed)
        ultimo=get_last_checkpoint(str(ckpt))
        epoca_previa=0.
        if ultimo:
            epoca_previa=float(json.loads((Path(ultimo)/'trainer_state.json').read_text())['epoch'])
        trainer=None;model=None
        try:
            if epoca_previa>=cfg['refit_epochs']-1e-6:
                assert abs(epoca_previa-cfg['refit_epochs'])<1e-6,'El checkpoint supera los epochs finales elegidos.'
                model=AutoModelForSequenceClassification.from_pretrained(ultimo,local_files_only=True).float()
                historial=json.loads((Path(ultimo)/'trainer_state.json').read_text())
            else:
                torch.set_float32_matmul_precision('high')  # mismo entrenamiento BF16 anterior
                model=ns['crear_modelo'](seed,cfg['classifier_dropout'],cfg)
                assert int(model.config.vocab_size)==len(tok),'El vocabulario base no coincide con el tokenizador original.'
                ds=ns['DatasetTokenizado'](train.MESSAGE_TRANSFORMER,train.IS_IRONIC)
                args=ns['argumentos_entrenamiento'](ckpt,cfg,seed,con_validacion=False,epochs=cfg['num_train_epochs'])
                trainer=ns['ClasificadorTrainer'](model=model,args=args,train_dataset=ds,
                    data_collator=DataCollatorWithPadding(tokenizer=tok,pad_to_multiple_of=8),
                    class_weights=ns['pesos_balanceados'](train.IS_IRONIC.to_numpy()),
                    label_smoothing=cfg['label_smoothing'],lr_decay=cfg['lr_decay'],
                    mascara_tokens=cfg['mascara_tokens'],callbacks=[ns['DetenerEnEpoca'](cfg['refit_epochs'])])
                print(f'{modelo} {v} semilla {seed}: entrenamiento final de {cfg["refit_epochs"]} epochs; máximo/scheduler {cfg["num_train_epochs"]}.',flush=True)
                trainer.train(resume_from_checkpoint=ultimo if ultimo else None)
                assert abs(float(trainer.state.epoch)-cfg['refit_epochs'])<1e-6
                historial=json.loads(trainer.state.to_json_string()) if hasattr(trainer.state,'to_json_string') else json.loads((Path(get_last_checkpoint(str(ckpt)))/'trainer_state.json').read_text())
            model.config.id2label={0:'NO_IRONICO',1:'IRONICO'}
            model.config.label2id={'NO_IRONICO':0,'IRONICO':1}
            if trainer is None:model.save_pretrained(destino,safe_serialization=True)
            else:trainer.save_model(str(destino))
            tok.save_pretrained(destino)
            nuevo=json.loads(json.dumps(meta));nuevo.update(seed=seed,
                fecha=datetime.now(timezone.utc).isoformat(),epocas_entrenadas=float(cfg['refit_epochs']),
                csv_train=str(ctx['datos']/'transformers'/f'train_transformers_{v}.csv'),
                sha256_csv_train=sha_archivo_fp32(ctx['datos']/'transformers'/f'train_transformers_{v}.csv'),
                versiones=ctx['versiones'],protocolo_id=ctx['protocolo_id'],
                revision_modelo_base=bases[modelo].name,origen=origen)
            escribir_json_fp32(nuevo,destino/'metadatos.json')
            escribir_json_fp32(historial,destino/'trainer_state.json')
        finally:
            del trainer,model
            gc.collect();torch.cuda.empty_cache()
    assert (destino/'model.safetensors').is_file()
    final={'protocolo_id':ctx['protocolo_id'],'modelo':modelo,'variante':v,'seed':seed,
        'config_id':meta['config_id'],'epochs':cfg['refit_epochs'],'origen':origen,
        'sha256_pesos':sha_archivo_fp32(destino/'model.safetensors'),
        'segundos':time.monotonic()-t0}
    escribir_json_fp32(final,destino/'completo_fp32.json')
    # Solo se eliminan checkpoints temporales generados por este flujo después de verificar
    # que el modelo final completo está guardado. Nunca se limpian las carpetas originales.
    if seed!=42:
        temporal=carpeta/'trabajos'/v/f'seed{seed}'/'ckpt'
        assert ctx['salida'] in temporal.resolve().parents
        shutil.rmtree(temporal,ignore_errors=True)
    return destino


def ejecutar_final_fp32(ctx):
    import torch
    from transformers import AutoTokenizer,AutoModelForSequenceClassification
    entorno_gpu_fp32(ctx)
    esperados=sum((ctx['datos']/f['carpeta']/'modelo_final'/v/'model.safetensors').stat().st_size
        for f in ctx['fuentes'].values() for v in VARIANTES_FP32)*5
    ya_guardados=sum(p.stat().st_size for p in ctx['salida'].glob('*/modelos_por_semilla/*/*/model.safetensors'))
    reserva=8*1024**3
    assert shutil.disk_usage(ctx['salida']).free>max(0,esperados-ya_guardados)+reserva,'Falta espacio para conservar los 45 modelos y los checkpoints temporales.'
    bases={};manifest=[]
    for modelo,f in ctx['fuentes'].items():
        carpeta=ctx['salida']/f['carpeta'];todas=[];exportadas=[]
        referencia=pd.read_csv(ctx['datos']/f['carpeta']/f'{f["prefijo"]}_test_predicciones_modelo_exportado.csv',dtype={'ID':str})
        for v in VARIANTES_FP32:
            src=ctx['datos']/f['carpeta']/'modelo_final'/v
            tok=AutoTokenizer.from_pretrained(src,local_files_only=True)
            train=comprobar_datos_tokenizados_fp32(ctx,modelo,v,tok)
            test=pd.read_csv(ctx['datos']/'transformers'/f'test_transformers_{v}.csv',dtype={'ID':str})
            ex=referencia[referencia.variante.eq(v)].set_index('ID').loc[test.ID]
            assert ex.y_true.to_numpy().tolist()==test.IS_IRONIC.tolist()
            if ex.texto.tolist()!=test.MESSAGE_TRANSFORMER.tolist():
                a=ex.texto.str.replace('\r\n','\n',regex=False).tolist()
                b=test.MESSAGE_TRANSFORMER.str.replace('\r\n','\n',regex=False).tolist()
                assert a==b,'El texto del test cambió respecto de la exportación original.'
                assert tok(ex.texto.tolist(),truncation=True,max_length=128,padding=False)['input_ids']==tok(test.MESSAGE_TRANSFORMER.tolist(),truncation=True,max_length=128,padding=False)['input_ids']
            propios=ex.coincide_train.astype(str).str.lower().eq('true').to_numpy()
            for seed in SEMILLAS_FINALES_FP32:
                destino=obtener_modelo_final_fp32(ctx,modelo,v,seed,tok,train,bases)
                modelo_disk=AutoModelForSequenceClassification.from_pretrained(destino,local_files_only=True).float().to('cuda').eval()
                try:probs=probabilidades_fp32(modelo_disk,tok,test.MESSAGE_TRANSFORMER.tolist(),16)
                finally:del modelo_disk;gc.collect();torch.cuda.empty_cache()
                pred=probs.argmax(axis=1)
                T=float(ctx['entradas'][(modelo,v)]['meta']['temperatura'])
                cal=np.log(np.clip(probs,1e-12,1))/T;cal-=cal.max(axis=1,keepdims=True)
                cal=np.exp(cal);cal/=cal.sum(axis=1,keepdims=True)
                d=pd.DataFrame(dict(variante=v,seed=seed,ID=test.ID.to_numpy(),y_true=test.IS_IRONIC.to_numpy(),
                    y_pred=pred,prob_ironia=probs[:,1],prob_ironia_calibrada=cal[:,1],coincide_train=propios))
                escribir_csv_fp32(d,carpeta/'predicciones_por_semilla'/f'{v}_seed{seed}_fp32.csv')
                todas.append(d)
                manifest.append(dict(modelo=modelo,variante=v,seed=seed,ruta=str(destino),
                    sha256_pesos=sha_archivo_fp32(destino/'model.safetensors')))
                escribir_csv_fp32(pd.DataFrame(manifest),ctx['salida']/'modelos_guardados.csv')
                if seed==42:
                    z=d.drop(columns='seed').copy()
                    z['texto']=test.MESSAGE_TRANSFORMER.to_numpy();z['mensaje_original']=test.MESSAGE.to_numpy()
                    z['tipo']=np.select([(z.y_true==1)&(z.y_pred==1),(z.y_true==0)&(z.y_pred==0),
                        (z.y_true==0)&(z.y_pred==1)],['VP','VN','FP'],default='FN')
                    exportadas.append(z)
                print(f'Finalizado: {modelo} {v} semilla {seed}. Modelos procesados: {len(manifest)}/45.',flush=True)
        full=pd.concat(todas,ignore_index=True)
        assert len(full)==9000 and full.groupby(['variante','seed']).size().eq(600).all()
        escribir_csv_fp32(full,carpeta/f'{f["prefijo"]}_test_probabilidades.csv')
        escribir_csv_fp32(pd.concat(exportadas,ignore_index=True),carpeta/f'{f["prefijo"]}_test_predicciones_modelo_exportado.csv')
    verificar_originales_fp32(ctx)
    escribir_json_fp32({'completo':True,'n_modelos':45,'n_entrenamientos_nuevos':36,
        'n_reutilizados':9,'precision_inferencia':'FP32','precision_entrenamiento':'BF16',
        'nota':'Nuevas ejecuciones finales de las semillas faltantes; no se afirma recuperar los pesos eliminados.'},ctx['salida']/'estado_entrenamiento_y_predicciones.json')
    return pd.DataFrame(manifest)


"""Recalcula la votación y las métricas con el mismo filtro para todos los modelos."""
from sklearn.metrics import f1_score,accuracy_score,precision_score,recall_score
from sklearn.metrics import roc_auc_score,average_precision_score,confusion_matrix

ORDEN_FP32=['LR','RF','XGBoost','HardVoting','BERT','DistilBERT','RoBERTuito','HardVotingTR']
NOMBRES_FP32={'LR':'RL','RF':'RF','XGBoost':'XGBoost','HardVoting':'Votación de ML',
    'BERT':'BERT','DistilBERT':'DistilBERT','RoBERTuito':'RoBERTuito','HardVotingTR':'Votación de Transformers'}
PAISES_FP32={'mx':'México','es':'España','cu':'Cuba'}


def reunir_predicciones_fp32(ctx,carpeta_transformers=None):
    datos=ctx['datos'];base=Path(carpeta_transformers or ctx['salida'])
    ml=pd.read_csv(datos/'resultados_ml_groupcv_tfidf_lr_rf_10k/predicciones_test_ml.csv',dtype={'ID':str})
    hv=pd.read_csv(datos/'resultados_ml_hard_voting_tfidf10k/hard_voting_test_predicciones.csv',dtype={'ID':str})
    hv['prob_ironia']=np.nan
    columnas=['variante','modelo','seed','ID','y_true','y_pred','prob_ironia']
    partes=[ml[columnas],hv[columnas]];transformers=[]
    for modelo,f in ctx['fuentes'].items():
        d=pd.read_csv(base/f['carpeta']/f'{f["prefijo"]}_test_probabilidades.csv',dtype={'ID':str})
        d['modelo']=modelo
        assert len(d)==9000 and not d.duplicated(['variante','seed','ID']).any()
        assert set(d.seed)==set(SEMILLAS_FINALES_FP32)
        partes.append(d[columnas]);transformers.append(d[columnas])
    tr=pd.concat(transformers,ignore_index=True)
    assert tr.groupby(['variante','seed','ID']).y_true.nunique().eq(1).all()
    votos=tr.pivot(index=['variante','seed','ID'],columns='modelo',values='y_pred')
    assert votos.shape==(9000,3) and not votos.isna().any().any()
    ironia=tr.drop_duplicates(['variante','seed','ID']).set_index(['variante','seed','ID']).y_true.loc[votos.index]
    vt=votos.reset_index()[['variante','seed','ID']]
    vt['y_pred']=votos.sum(axis=1).ge(2).astype(int).to_numpy()
    vt['y_true']=ironia.to_numpy();vt['modelo']='HardVotingTR';vt['prob_ironia']=np.nan
    partes.append(vt[columnas]);full=pd.concat(partes,ignore_index=True)
    assert len(full)==72000 and not full.duplicated(['variante','modelo','seed','ID']).any()
    referencias={}
    for (v,m,s),g in full.groupby(['variante','modelo','seed']):
        assert len(g)==600 and set(g.y_true)=={0,1} and set(g.y_pred)<={0,1}
        etiqueta=g.set_index('ID').y_true.sort_index()
        if v in referencias:assert etiqueta.equals(referencias[v])
        else:referencias[v]=etiqueta
    full['coincide_train']=[(v,i) in ctx['ids_comunes'] for v,i in zip(full.variante,full.ID)]
    return full


def calcular_comparacion_fp32(ctx,carpeta_transformers=None):
    full=reunir_predicciones_fp32(ctx,carpeta_transformers)
    out=ctx['salida']/'resultados_comparacion_final';out.mkdir(parents=True,exist_ok=True)
    escribir_csv_fp32(full,out/'predicciones_test_completo.csv')
    escribir_csv_fp32(pd.DataFrame(sorted(ctx['ids_comunes']),columns=['variante','ID']),out/'ids_test_coincidentes_con_train.csv')
    filas=[];conf=[]
    for (v,m,s),g in full.groupby(['variante','modelo','seed']):
        a=g[~g.coincide_train]
        assert len(a)==TAMANO_TEST_FP32[v]
        y=a.y_true;p=a.y_pred;prob=a.prob_ironia
        tiene_prob=prob.notna().all()
        filas.append(dict(variante=v,modelo=m,seed=int(s),n_test=len(a),
            f1_macro=f1_score(y,p,average='macro'),accuracy=accuracy_score(y,p),
            precision_macro=precision_score(y,p,average='macro',zero_division=0),
            recall_macro=recall_score(y,p,average='macro',zero_division=0),
            f1_ironia=f1_score(y,p,pos_label=1,zero_division=0),
            f1_macro_sin_coincidencias=f1_score(y,p,average='macro'),
            f1_macro_test_completo=f1_score(g.y_true,g.y_pred,average='macro'),
            auc_roc=roc_auc_score(y,prob) if tiene_prob else np.nan,
            average_precision=average_precision_score(y,prob) if tiene_prob else np.nan))
        vn,fp,fn,vp=confusion_matrix(y,p,labels=[0,1]).ravel()
        conf.append(dict(variante=v,modelo=m,seed=int(s),VP=vp,VN=vn,FP=fp,FN=fn,
            tasa_FP=fp/(fp+vn),tasa_FN=fn/(fn+vp)))
    met=pd.DataFrame(filas);con=pd.DataFrame(conf)
    metricas=['f1_macro','accuracy','precision_macro','recall_macro','f1_ironia',
        'f1_macro_sin_coincidencias','f1_macro_test_completo','auc_roc','average_precision']
    summary=met.groupby(['variante','modelo'])[metricas].agg(['mean','std'])
    summary.columns=[f'{a}_{b}' for a,b in summary.columns];summary=summary.reset_index()
    summary['n_semillas']=5
    summary['n_test']=summary.variante.map(TAMANO_TEST_FP32)
    summary=summary.assign(_v=summary.variante.map({v:i for i,v in enumerate(VARIANTES_FP32)}),
        _m=summary.modelo.map({m:i for i,m in enumerate(ORDEN_FP32)})).sort_values(['_v','_m']).drop(columns=['_v','_m']).reset_index(drop=True)
    errores=con.groupby(['variante','modelo'])[['VP','VN','FP','FN','tasa_FP','tasa_FN']].mean().reset_index()
    for n in ['FP','FN']:
        std=con.groupby(['variante','modelo'])[n].std().rename(n+'_std').reset_index()
        errores=errores.merge(std,on=['variante','modelo'],validate='one_to_one')
    ganadores=[]
    for v in VARIANTES_FP32:
        for familia,opciones in [('ML',['LR','RF','XGBoost','HardVoting']),
                                ('Transformers',['BERT','DistilBERT','RoBERTuito','HardVotingTR'])]:
            g=summary[summary.variante.eq(v)&summary.modelo.isin(opciones)].sort_values(['f1_macro_mean','modelo'],ascending=[False,True])
            ganador=g.iloc[0]
            ganadores.append(dict(variante=v,familia=familia,modelo=ganador.modelo,f1_macro=ganador.f1_macro_mean))
    ganar=pd.DataFrame(ganadores)
    def formato(row,n):return f'{row[n+"_mean"]:.3f} ± {row[n+"_std"]:.3f}'
    tab1=pd.DataFrame({'Variante':summary.variante.map(PAISES_FP32),'Modelo':summary.modelo.map(NOMBRES_FP32),
        **{etiqueta:summary.apply(lambda row,n=nombre:formato(row,n),axis=1) for nombre,etiqueta in
            [('f1_macro','F1-Macro'),('precision_macro','Precision macro'),('recall_macro','Recall macro'),('f1_ironia','F1 de la clase irónica')]}})
    pro=summary[summary.auc_roc_mean.notna()]
    tab2=pd.DataFrame({'Variante':pro.variante.map(PAISES_FP32),'Modelo':pro.modelo.map(NOMBRES_FP32),
        'AUC-ROC':pro.apply(lambda row:formato(row,'auc_roc'),axis=1),
        'AP':pro.apply(lambda row:formato(row,'average_precision'),axis=1)})
    sens=summary[['variante','modelo','f1_macro_mean','f1_macro_std','f1_macro_test_completo_mean','f1_macro_test_completo_std']].copy()
    sens['diferencia_filtrado_menos_completo']=sens.f1_macro_mean-sens.f1_macro_test_completo_mean
    for nombre,tabla in [('metricas_test_por_semilla.csv',met),('tabla_resumen_test.csv',summary),
        ('tabla_fp_fn_test.csv',errores),('ganadores_familia_variante.csv',ganar),
        ('tabla_6_1_metricas.csv',tab1),('tabla_6_2_auc_ap.csv',tab2),('sensibilidad_filtrado.csv',sens)]:
        escribir_csv_fp32(tabla,out/nombre)
    antes=pd.read_csv(ctx['datos']/'resultados_comparacion_final/tabla_resumen_test.csv')
    antes['modelo']=antes.modelo.replace({'BETO':'BERT','DistilBETO':'DistilBERT'})
    diferencias=summary.merge(antes[['variante','modelo','f1_macro_mean']],on=['variante','modelo'],suffixes=('_fp32','_anterior'),validate='one_to_one')
    diferencias['diferencia_f1']=diferencias.f1_macro_mean_fp32-diferencias.f1_macro_mean_anterior
    escribir_csv_fp32(diferencias,out/'cambios_frente_evaluacion_anterior.csv')
    # El procedimiento de ML no cambió: sus resultados deben conservarse exactamente.
    assert diferencias[diferencias.modelo.isin(['LR','RF','XGBoost','HardVoting'])].diferencia_f1.abs().lt(1e-12).all()
    escribir_json_fp32({'completo':True,'n_predicciones':len(full),'n_metricas_por_semilla':len(met),
        'test_filtrado':TAMANO_TEST_FP32,'ML_sin_cambios':True},out/'estado_comparacion.json')
    return summary,ganar,full


def matrices_articulo_fp32(ctx,ganadores,full):
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize
    limpio=full[~full.coincide_train]
    fig,axes=plt.subplots(2,3,figsize=(10,6.1),squeeze=False)
    for j,v in enumerate(VARIANTES_FP32):
        for i,familia in enumerate(['Transformers','ML']):
            modelo=ganadores[ganadores.variante.eq(v)&ganadores.familia.eq(familia)].iloc[0].modelo
            g=limpio[limpio.variante.eq(v)&limpio.modelo.eq(modelo)]
            mat=np.mean([confusion_matrix(z.y_true,z.y_pred,labels=[0,1]) for _,z in g.groupby('seed')],axis=0)
            porcentajes=100*mat/mat.sum(axis=1,keepdims=True)
            ax=axes[i,j];ax.imshow(porcentajes,cmap='Blues',norm=Normalize(0,100))
            for a in range(2):
                for b in range(2):ax.text(b,a,f'{porcentajes[a,b]:.1f} %',ha='center',va='center',fontsize=12,color='white' if porcentajes[a,b]>55 else 'black')
            ax.set_xticks([0,1],['No irónico','Irónico']);ax.set_yticks([0,1],['No irónico','Irónico'])
            ax.set_title(f'{PAISES_FP32[v]} · {NOMBRES_FP32[modelo]}',fontsize=10)
            ax.set_xlabel('Predicción')
            if j==0:ax.set_ylabel('Clase real')
    fig.tight_layout()
    out=ctx['salida']/'resultados_comparacion_final'
    for ext in ['png','svg']:fig.savefig(out/f'figura_6_1_matrices_confusion.{ext}',dpi=300,bbox_inches='tight',facecolor='white')
    plt.close(fig)


"""Explicaciones de semilla 42 con la misma función de inferencia que el test final."""
import unicodedata

CASOS_LOCALES_FP32={'mx':'7848dd4a3d97f9e1ec61b927789ebf18',
    'es':'987610faf6fde801af91fb59f62f38c9','cu':'9a95b2dc61f5a657f2794e2d4995e22d'}


def agrupar_palabras_fp32(tabla):
    filas=[]
    for (_, _),g in tabla.sort_values('posicion').groupby(['variante','ID'],sort=False):
        actual=None;anterior='';k=0
        for r in g.itertuples():
            tok=str(r.token)
            if actual is None or tok[:1].isspace() or anterior[-1:].isspace():
                if actual is not None:filas.append(actual)
                actual=dict(variante=r.variante,ID=r.ID,tipo=r.tipo,y_true=r.y_true,y_pred=r.y_pred,
                    prob_ironia=r.prob_ironia,posicion=k,token=tok,shap=float(r.shap),n_tokens=1)
                k+=1
            else:
                actual['token']+=tok;actual['shap']+=float(r.shap);actual['n_tokens']+=1
            anterior=tok
        if actual is not None:filas.append(actual)
    d=pd.DataFrame(filas)
    d['token']=d.token.str.strip()
    bordes='"\'“”«»¿?¡!.,;:()[]…'
    d['token_normalizado']=d.token.map(lambda s:s.lower().strip(bordes) or s.lower())
    return d[d.token_normalizado.ne('')]


def calcular_shap_final_fp32(ctx):
    import torch,shap
    from transformers import AutoTokenizer,AutoModelForSequenceClassification
    especiales={'','[CLS]','[SEP]','[PAD]','<s>','</s>','<pad>'}
    metadatos_todos=[]
    for modelo,f in ctx['fuentes'].items():
        carpeta=ctx['salida']/f['carpeta'];pref=f['prefijo']
        p=pd.read_csv(carpeta/f'{pref}_test_predicciones_modelo_exportado.csv',dtype={'ID':str})
        partes=[];partes_extra=[]
        for v in VARIANTES_FP32:
            ruta=carpeta/'modelos_por_semilla/seed42'/v
            assert comprobar_modelo_completo_fp32(ctx,ruta,modelo,v,42)
            mod=AutoModelForSequenceClassification.from_pretrained(ruta,local_files_only=True).float().to('cuda').eval()
            tok=AutoTokenizer.from_pretrained(ruta,local_files_only=True)
            pred=p[p.variante.eq(v)&~p.coincide_train.astype(str).str.lower().eq('true')]
            muestra=pd.concat([g.sample(min(50,len(g)),random_state=42) for _,g in pred.groupby('tipo')],ignore_index=True)
            muestras_dir=carpeta/'shap_fp32';muestras_dir.mkdir(exist_ok=True)
            escribir_csv_fp32(muestra,muestras_dir/f'{v}_muestra.csv')
            def funcion(textos):
                textos=[textos] if isinstance(textos,str) else list(textos)
                return probabilidades_fp32(mod,tok,textos,16)
            # Misma escala de probabilidad y máscara de texto que el análisis previo.
            # La función de predicción es ahora explícitamente la usada en la evaluación FP32.
            explicador=shap.Explainer(funcion,shap.maskers.Text(tok),algorithm='partition',
                output_names=['NO_IRONICO','IRONICO'],seed=42)
            solicitados=muestra.copy();solicitados['extra_local']=False
            if modelo=='RoBERTuito' and CASOS_LOCALES_FP32[v] not in set(muestra.ID):
                extra=pred[pred.ID.eq(CASOS_LOCALES_FP32[v])].copy()
                assert len(extra)==1,'El ejemplo local anterior ya no está disponible tras el filtro; revisar la selección.'
                extra['extra_local']=True;solicitados=pd.concat([solicitados,extra],ignore_index=True)
            for numero,r in enumerate(solicitados.itertuples(),1):
                cache=muestras_dir/'cache'/v/f'{r.ID}.json'
                if cache.exists():
                    j=json.loads(cache.read_text(encoding='utf-8'))
                    assert j['protocolo_id']==ctx['protocolo_id'] and j['texto']==r.texto
                    assert j['y_pred']==int(r.y_pred) and abs(j['prob_ironia']-float(r.prob_ironia))<1e-12
                else:
                    base=funcion([r.texto])[0]
                    assert base.argmax()==int(r.y_pred),f'{modelo}/{v}/{r.ID}: la clase cambia con el lote; revisar antes de explicar.'
                    assert abs(base[1]-float(r.prob_ironia))<1e-6
                    valores=explicador([r.texto],batch_size=32,max_evals=500,silent=True)
                    vals=np.asarray(valores.values[0])[:,1]
                    valor_base=float(np.asarray(valores.base_values)[0,1])
                    assert abs(valor_base+vals.sum()-base[1])<1e-5,'La explicación no reconstruye la probabilidad.'
                    j={'protocolo_id':ctx['protocolo_id'],'texto':r.texto,'y_pred':int(r.y_pred),
                        'prob_ironia':float(r.prob_ironia),'tokens':[str(t) for t in valores.data[0]],
                        'contribuciones':vals.tolist(),'valor_base':valor_base,
                        'error_reconstruccion':float(abs(valor_base+vals.sum()-base[1]))}
                    escribir_json_fp32(j,cache)
                filas=[]
                for pos,(token,valor) in enumerate(zip(j['tokens'],j['contribuciones'])):
                    if token.strip() in especiales:
                        assert abs(valor)<1e-8;continue
                    filas.append(dict(variante=v,ID=r.ID,tipo=r.tipo,y_true=int(r.y_true),y_pred=int(r.y_pred),
                        prob_ironia=float(r.prob_ironia),posicion=pos,token=token,
                        token_normalizado=token.replace('Ġ',' ').replace('##','').strip().lower(),shap=valor))
                (partes_extra if r.extra_local else partes).append(pd.DataFrame(filas))
                metadatos_todos.append(dict(modelo=modelo,variante=v,ID=r.ID,tipo=r.tipo,
                    extra_local=bool(r.extra_local),error_reconstruccion=j['error_reconstruccion']))
                print(f'SHAP {modelo} {v}: {numero}/{len(solicitados)} textos.',flush=True)
            del explicador,mod;gc.collect();torch.cuda.empty_cache()
        tokens=pd.concat(partes,ignore_index=True)
        palabras=agrupar_palabras_fp32(tokens)
        escribir_csv_fp32(tokens,carpeta/f'{pref}_shap_tokens.csv')
        escribir_csv_fp32(palabras,carpeta/f'{pref}_shap_palabras.csv')
        global_=palabras.groupby(['variante','token_normalizado']).agg(n_textos=('ID','nunique'),
            shap_medio=('shap','mean'),shap_abs_medio=('shap',lambda s:s.abs().mean()),shap_total=('shap','sum')).reset_index()
        global_=global_[global_.n_textos.ge(5)].sort_values(['variante','shap_medio'],ascending=[True,False])
        escribir_csv_fp32(global_,carpeta/f'{pref}_shap_global.csv')
        # Los casos añadidos solo para las ilustraciones no cambian las medias globales.
        extras=pd.concat(partes_extra,ignore_index=True) if partes_extra else tokens.iloc[:0]
        if not extras.empty:
            escribir_csv_fp32(agrupar_palabras_fp32(extras),carpeta/f'{pref}_shap_palabras_locales.csv')
    detalle=pd.DataFrame(metadatos_todos)
    escribir_csv_fp32(detalle,ctx['salida']/'verificacion_shap_fp32.csv')
    assert detalle.error_reconstruccion.lt(1e-5).all()
    verificar_originales_fp32(ctx)
    escribir_json_fp32({'completo':True,'explicacion':'PartitionSHAP de probabilidades FP32',
        'n_casos_muestra':int((~detalle.extra_local).sum()),'n_extras_locales':int(detalle.extra_local.sum()),
        'precision_inferencia':'La misma función probabilidades_fp32 del test',
        'muestreo':'semilla 42, hasta 50 por tipo de resultado, filtro propio del modelo'},ctx['salida']/'estado_shap.json')
    return detalle


"""Genera figuras en la carpeta nueva; el artículo se revisa después de la ejecución."""
def figuras_shap_articulo_fp32(ctx,ganadores,fuentes_figuras):
    import matplotlib.pyplot as plt
    nuevos=ganadores[ganadores.familia.eq('Transformers')]
    if not nuevos.modelo.eq('RoBERTuito').all():
        print('Cambió el ganador Transformer de alguna variante. Revisar la selección antes de regenerar las figuras SHAP del artículo.')
        return False
    if not (ctx['datos']/'resultados_interpretabilidad/votacion_ml/shap_votacion_global.csv').exists():
        print('Falta el CSV SHAP de la votación de ML en este equipo. Las nuevas explicaciones Transformer ya están guardadas; las figuras globales podrán generarse donde se conserve ese CSV.')
        return False
    antes=os.environ.get('SHAP_OUTPUT_DIR');directorio=Path.cwd()
    os.environ['SHAP_OUTPUT_DIR']=str(ctx['salida']/'resultados_interpretabilidad')
    ns={'RAIZ_FP32':ctx['salida']}
    try:
        os.chdir(ctx['raiz'])
        exec(compile(fuentes_figuras['configuracion'],'configuracion_figuras','exec'),ns)
        # ML y su votación conservan sus valores originales; sustituir solo los Transformers.
        partes=[ns['GLOBAL'][ns['GLOBAL'].familia.eq('ML')]]
        for modelo,f in ctx['fuentes'].items():
            g=pd.read_csv(ctx['salida']/f['carpeta']/f'{f["prefijo"]}_shap_global.csv')
            g=g.rename(columns={'token_normalizado':'termino'}).assign(modelo=modelo,familia='Transformer')
            g['termino_norm']=g.termino.map(ns['normalizar'])
            partes.append(g[['variante','modelo','familia','termino','n_textos','shap_medio','shap_abs_medio','termino_norm']])
        ns['GLOBAL']=pd.concat(partes,ignore_index=True)
        globales=fuentes_figuras['global'].replace(
            'ruta_test = DATA_DIR / "resultados_comparacion_final/tabla_resumen_test.csv"',
            'ruta_test = RAIZ_FP32 / "resultados_comparacion_final/tabla_resumen_test.csv"')
        assert globales!=fuentes_figuras['global']
        ns['display']=lambda *_:None
        exec(compile(globales,'figura_global_fp32','exec'),ns)
        local=fuentes_figuras['local'].replace(
            'carpeta = DATA_DIR / f"resultados_{prefijo}_grouped_5x3_tpe20" if prefijo else ML_DIR',
            'carpeta = RAIZ_FP32 / f"resultados_{prefijo}_grouped_5x3_tpe20" if prefijo else ML_DIR')
        assert local!=fuentes_figuras['local']
        objetivo='''        palabras = pd.read_csv(carpeta / f"{prefijo}_shap_palabras.csv", dtype={"ID": str})'''
        assert objetivo in local
        local=local.replace(objetivo,objetivo+'''
        extra = carpeta / f"{prefijo}_shap_palabras_locales.csv"
        if extra.exists():
            palabras = pd.concat([palabras, pd.read_csv(extra, dtype={"ID": str})], ignore_index=True).drop_duplicates(["variante", "ID", "posicion"])''')
        exec(compile(local,'figura_local_fp32','exec'),ns)
        fig=ns['EJES_LOCALES'][0].figure;fig.canvas.draw();renderer=fig.canvas.get_renderer()
        for previo,actual in zip(ns['EJES_LOCALES'],ns['EJES_LOCALES'][1:]):
            assert min(t.get_window_extent(renderer).x0 for t in actual.get_yticklabels())>previo.get_window_extent(renderer).x1+2
        return True
    finally:
        os.chdir(directorio)
        if antes is None:os.environ.pop('SHAP_OUTPUT_DIR',None)
        else:os.environ['SHAP_OUTPUT_DIR']=antes


def cerrar_ejecucion_fp32(ctx,ganadores,shap_completo,figuras_shap):
    verificar_originales_fp32(ctx)
    modelos=pd.read_csv(ctx['salida']/'modelos_guardados.csv')
    assert len(modelos)==45 and modelos.groupby(['modelo','variante']).seed.nunique().eq(5).all()
    for r in modelos.itertuples():
        assert sha_archivo_fp32(Path(r.ruta)/'model.safetensors')==r.sha256_pesos
    estado_comp=json.loads((ctx['salida']/'resultados_comparacion_final/estado_comparacion.json').read_text())
    assert estado_comp['completo'] and estado_comp['n_predicciones']==72000
    estado_shap=json.loads((ctx['salida']/'estado_shap.json').read_text())
    assert shap_completo and estado_shap['completo']
    texto=['# Ejecución final unificada en FP32','',
        'Se conservaron las configuraciones finales y los epochs ya seleccionados. Se reutilizaron los nueve modelos de semilla 42 y se realizaron 36 entrenamientos finales nuevos. No se repitieron la búsqueda de hiperparámetros ni nested CV. Los 45 modelos y sus huellas quedan guardados.','',
        'El entrenamiento conserva BF16. La evaluación final y SHAP utilizan la misma función de inferencia FP32, sin autocast y con matmul highest. El test no selecciona configuraciones, epochs ni semillas.','',
        'Las semillas 52, 62, 72 y 82 corresponden a nuevas ejecuciones finales, no a una recuperación de los pesos borrados. Por tanto, sus diferencias frente a las tablas anteriores no deben atribuirse únicamente a precisión numérica.','',
        '## Archivos para revisar','',
        '- resultados_comparacion_final/tabla_6_1_metricas.csv y tabla_6_2_auc_ap.csv: tablas del artículo.',
        '- resultados_comparacion_final/sensibilidad_filtrado.csv: mismo conjunto de predicciones, test filtrado frente a completo.',
        '- resultados_comparacion_final/ganadores_familia_variante.csv y figura_6_1_matrices_confusion.png: ganadores del test por familia y variante.',
        '- resultados_comparacion_final/cambios_frente_evaluacion_anterior.csv: contraste de las ejecuciones; no sirve para elegir la más favorable.',
        '- modelos_guardados.csv: los 45 modelos, con sus rutas y huellas.',
        '- verificacion_shap_fp32.csv: reconstrucción de probabilidades en cada explicación.',
        '- resultados_interpretabilidad/articulo/: figuras SHAP y contribuciones locales, si se mantuvo RoBERTuito como ganador Transformer en las tres variantes.','',
        '## Estado de SHAP','',
        f'Explicaciones completadas: {shap_completo}. Figuras del artículo regeneradas: {figuras_shap}.',
        'Las muestras mantienen el filtro propio y hasta 50 textos por resultado. Los ejemplos locales anteriores que no entren en la muestra se explican aparte y no modifican las medias globales.','',
        '## Trasladar al artículo','',
        'Revisar las nuevas tablas y patrones antes de actualizar Resultados, Discusión y Conclusiones. La sección de nested CV y las configuraciones no cambian. En Metodología, reemplazar la aclaración BF16/FP32 anterior por la precisión uniforme de inferencia final y describir la nueva ejecución de las semillas faltantes.','',
        'Al adoptar esta ejecución, reportarla completa, incluso si alguna métrica resulta menor. No combinar las semillas más favorables de los dos procedimientos. Los resultados y archivos originales se conservaron sin cambios.']
    (ctx['salida']/'LEER_RESULTADO_FINAL.md').write_text('\n'.join(texto),encoding='utf-8')
    escribir_json_fp32({'entrenamiento_y_predicciones_completos':True,'comparacion_completa':True,
        'shap_completo':bool(shap_completo),'figuras_shap_completas':bool(figuras_shap),
        'n_modelos':45,'originales_sin_cambios':True,'protocolo_id':ctx['protocolo_id']},ctx['salida']/'estado_final.json')
    print('Ejecución terminada. Abre LEER_RESULTADO_FINAL.md en:',ctx['salida'])


## 3. Verificación y plan de los 45 modelos

Verifica datos, configuraciones, pesos originales y el filtro común. La verificación de saltos de línea distingue diferencias de serialización de cambios en los tokens que recibe el modelo.

In [ ]:
CONTEXTO = preparar_final_fp32(RAIZ_PROYECTO, SALIDA_FP32, FUENTES_MODELOS)
display(pd.DataFrame(CONTEXTO["plan"]))

## 4. Entrenamientos finales pendientes y predicciones FP32

Esta celda entrena únicamente las 36 configuraciones finales pendientes, reutiliza los nueve modelos de semilla 42 y guarda todos los pesos. Los checkpoints temporales se eliminan solo después de comprobar que el modelo final correspondiente está exportado. Las predicciones se calculan después de recargar los pesos, sin la precisión mixta del entrenador.

In [ ]:
MODELOS_GUARDADOS = ejecutar_final_fp32(CONTEXTO)
display(MODELOS_GUARDADOS)

## 5. Votación, métricas y matrices de confusión

Combina las predicciones de la misma semilla. Usa el test filtrado común, conserva las cinco semillas y calcula sensibilidad con las mismas predicciones sobre los 600 textos. La evaluación de ML permanece como estaba.

In [ ]:
RESUMEN_FINAL, GANADORES, PREDICCIONES = calcular_comparacion_fp32(CONTEXTO)
matrices_articulo_fp32(CONTEXTO, GANADORES, PREDICCIONES)
display(RESUMEN_FINAL)
display(GANADORES)

## 6. SHAP con la misma función de predicción FP32

Explica los modelos de semilla 42 con hasta 50 textos por variante y tipo de resultado, con el filtro propio del modelo. Comprueba que la referencia y las contribuciones reconstruyen la probabilidad. Guarda cada caso para continuar sin repetirlo. Los ejemplos locales adicionales se mantienen fuera de las medias globales.

In [ ]:
VERIFICACION_SHAP = calcular_shap_final_fp32(CONTEXTO)
display(VERIFICACION_SHAP.groupby(["modelo", "variante"]).agg(textos=("ID", "nunique"), error_maximo=("error_reconstruccion", "max")))

## 7. Figuras SHAP para revisar el artículo

Reutiliza las explicaciones de ML y su votación y sustituye solo los datos Transformer por los nuevos. Si cambia el ganador Transformer de alguna variante, detiene la selección automática de estas figuras y lo indica para revisar el enfoque del artículo. No sustituye el borrador ni sus imágenes anteriores.

In [ ]:
FIGURAS_SHAP_LISTAS = figuras_shap_articulo_fp32(CONTEXTO, GANADORES, FUENTES_FIGURAS)

## 8. Comprobación final y archivos para el artículo

In [ ]:
cerrar_ejecucion_fp32(CONTEXTO, GANADORES, shap_completo=True, figuras_shap=FIGURAS_SHAP_LISTAS)

## Al terminar

Abrir `LEER_RESULTADO_FINAL.md` en la carpeta nueva. Los archivos principales para revisar son:

- `estado_final.json`: confirma qué etapas se completaron.
- `resultados_comparacion_final/tabla_6_1_metricas.csv` y `tabla_6_2_auc_ap.csv`.
- `resultados_comparacion_final/sensibilidad_filtrado.csv` y `ganadores_familia_variante.csv`.
- `modelos_guardados.csv`: rutas y huellas de los 45 modelos.
- `verificacion_shap_fp32.csv`: reconstrucción de las probabilidades explicadas.
- `resultados_interpretabilidad/articulo/`: nuevas figuras, si la selección vigente sigue siendo aplicable.

Esta nueva ejecución debe revisarse antes de actualizar la redacción de Resultados y las conclusiones. Los valores de nested CV y las configuraciones finales no se recalculan aquí. Los resultados anteriores no se mezclan con los nuevos ni se sustituye cada semilla por su versión más favorable.
